# MB2 · Cooccurrences, PMI et embeddings — notebook d'exercices

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cemah2/workbookMaths/blob/main/chapitres/mb2_cooccurrences/03_notebook.ipynb)

La partie 0 vérifie tes exercices papier. Chaque exercice de code : un énoncé, une cellule à compléter (les `...` et les `raise NotImplementedError`), puis une cellule de vérification (`wbm.check`). Dans les parties A à D, donne les valeurs que rend le calcul, **sans les arrondir** : les décimales demandées sont celles de ta copie. « Tout exécuter » (*Run all*) va jusqu'au bout même si rien n'est rempli : ce qui n'est pas fait affiche ⏳. Bloqué 15 minutes ? `04_indices.md`.

> Travaille dans **ta copie** (`mon_travail/mb2_cooccurrences/03_notebook.ipynb`, créée par `python tools/start_chapter.py MB2`) ; la version du dépôt est mise à jour par Claude. Sur Colab, le badge ouvre la version du dépôt, qui n'est pas enregistrée : crée puis ouvre ta copie comme l'explique `00_setup/COLAB.md` §2.

| Partie | ID | Titre | Type | ★ | ⏱️ |
|---|---|---|---|---|---|
| 0 | MB2.1 à MB2.7, MB2.11 | Les exercices papier (vérification) | ✏️ 🔗 | | |
| A | MB2.13 | La table des PMI d'une loi jointe, −∞ sur les cases vides · `pmi` | 🔨 | ★ | 30 |
|  | MB2.14 | Deux briques : le logit d'une population estimé sur un échantillon cas-témoins, et retirer une direction des lignes d'une matrice · `case_control_logit`, `remove_direction` | 🔨 | ★★ | 35 |
| B | MB2.15 | Prédire puis mesurer la dispersion de la PMI estimée sur 400 observations, pour une case rare et une case fréquente, avec et sans lissage | 🔮 | ★★ | 30 |
|  | MB2.16 | Des magasins dans le plan des deux facteurs U₂S₂^β, pour β = 0, ½ et 1 : ce qui s'étire, et les angles qui changent | 🎨 | ★ | 35 |
| C | MB2.17 | Chasse à l'erreur : une PMI calculée sans le total, des facteurs U S^β et V S^β, une direction retirée sans diviser par u·u | 🐛 | ★ | 30 |
|  | MB2.18 | Un test de propriété de `pmi` sur des tables tirées au hasard : la borne, la transposition, et la moyenne qui redonne l'information mutuelle de M06 | 🛠️ | ★ | 35 |
| D | MB2.19 | Défi : retrouver, sur 3 000 observations, les cinq paires les plus associées d'une table cachée de 10 × 10, sans se laisser piéger par les cases rares | 🏆 | ★★★ | 45 |

In [ ]:
# ⚙️ SETUP: run this cell first (on Colab or on your computer)
FAST_MODE = True  # True = quick version (a few seconds per cell); False = bigger experiments
SEED = 42

import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/cemah2/workbookMaths.git"
DRIVE_DIR = Path("/content/drive/MyDrive/workbookMaths")

try:
    import google.colab  # noqa: F401  (this module only exists on Colab)
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")
    subprocess.run(["git", "config", "--global", "--add", "safe.directory", str(DRIVE_DIR)])
    if not (DRIVE_DIR / ".git").exists():
        subprocess.run(["git", "clone", REPO_URL, str(DRIVE_DIR)], check=True)
    else:
        # Get the new chapters. Your own work only lives in mon_travail/, which Claude never
        # touches, so "rebase + autostash" keeps it safe (even with local commits).
        identity = subprocess.run(["git", "-C", str(DRIVE_DIR), "config", "user.email"],
                                  capture_output=True).returncode == 0
        who = [] if identity else ["-c", "user.name=Workbook", "-c", "user.email=workbook@localhost"]
        pull = subprocess.run(["git", *who, "-C", str(DRIVE_DIR), "pull", "--rebase", "--autostash"],
                              capture_output=True, text=True)
        conflicts = subprocess.run(["git", "-C", str(DRIVE_DIR), "diff", "--name-only", "--diff-filter=U"],
                                   capture_output=True, text=True).stdout.split()
        if conflicts:  # a file changed outside mon_travail/ AND by Claude: git leaves conflict markers in it
            print("⚠️ Tes modifications de " + ", ".join(conflicts) + " (hors de mon_travail/) entrent en conflit "
                  "avec la nouvelle version ; elles sont gardées dans `git stash`. Pour reprendre la version du "
                  "dépôt : git restore --source=HEAD --staged --worktree <fichier> (00_setup/COLAB.md, « Problèmes "
                  "fréquents »).")
        elif pull.returncode != 0:
            print("⚠️ git pull a échoué (voir 00_setup/COLAB.md, « Problèmes fréquents ») :\n" + pull.stderr)
    ROOT = DRIVE_DIR
else:
    ROOT = Path(os.environ.get("WBM_ROOT", Path.cwd())).resolve()
    while not (ROOT / "src" / "wbm").exists() and ROOT != ROOT.parent:
        ROOT = ROOT.parent
    if not (ROOT / "src" / "wbm").exists():
        raise FileNotFoundError("Dépôt introuvable : ouvre ce notebook depuis le dossier de workbookMaths.")

os.environ["WBM_ROOT"] = str(ROOT)
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

import wbm  # noqa: E402

cfg = wbm.setup(seed=SEED, fast=FAST_MODE)
FAST_MODE = cfg.fast
mymath = wbm.load_mymath("learner", missing_ok=True, fallback="ref", chapter="MB2")  # your own library

## Objectifs et rappel express

- Vérifier tes calculs papier (partie 0) : des entiers en base 2 et en base 16, la PMI d'une table d'oiseaux et son lissage, les facteurs d'une SVD tronquée, le logit d'une bannière sous-échantillonnée, une analogie entre aliments, une direction retirée de mesures d'animaux, et un pont vers la moyenne masquée.
- Écrire les trois fonctions de `mymath/cooccurrence.py` (`pmi`, `case_control_logit`, `remove_direction`) et les tester.
- Prédire puis mesurer la dispersion d'une PMI estimée, dessiner des magasins dans le plan de deux facteurs, corriger trois erreurs classiques, écrire un test de propriété, et relever un défi.

**Rappel express.** $\operatorname{PMI}(x, y) = \ln\frac{p(x, y)}{p(x)\,p(y)}$, en nats, $-\infty$ sur une case vide ; $\operatorname{PMI}(x, y) \le \min(-\ln p(x), -\ln p(y))$ ; $I(X ; Y) = \sum p(x, y)\operatorname{PMI}(x, y)$ sur les cases non vides. Lissage : $p_\alpha(y) \propto n(y)^\alpha$, la loi tempérée de $T = 1/\alpha$, qui décale chaque colonne de $-\ln\frac{p_\alpha(y)}{p(y)}$. Facteurs : $(\mathbf{U}_k\mathbf{S}_k^\beta)(\mathbf{V}_k\mathbf{S}_k^{1-\beta})^\top = \mathbf{M}_k$ pour tout $\beta$. Cas-témoins : la cote est multipliée par $s_1/s_0$, le logit décalé de $\ln\frac{s_1}{s_0}$. Retirer $\mathbf{u}$ : $\mathbf{x} - \frac{\mathbf{x}\cdot\mathbf{u}}{\mathbf{u}\cdot\mathbf{u}}\,\mathbf{u}$.

## Partie 0 · Vérifier tes exercices papier (✏️ MB2.1 à MB2.7 ; 🔗 MB2.11)

Fais d'abord les exercices de `02_exercices.md` **sur papier**, dans ta copie de `06_mes_reponses.md`. Reporte ensuite chaque réponse ici : **la valeur** que tu as trouvée, sous la forme indiquée en commentaire, jamais le calcul qui y mène :
- un entier, ou un nombre avec un point décimal, `-1.25`, arrondi comme le demande la question (« a number » sans décimales : la valeur exacte, ces calculs tombent juste) ;
- une fraction irréductible entre guillemets, `"5/8"` ;
- un nombre écrit en base 2 ou en base 16 comme un texte entre guillemets, avec ses seuls chiffres, `"1011"` ou `"3F"` (sans `0b` ni `0x`) ;
- une liste `[0.2, -1]`, dans l'ordre que dit la question ; des espèces comme la liste de leurs numéros ; une matrice comme la liste de ses **lignes**, `[[1, 0], [-2, 3]]` ;
- `True` ou `False` pour un vrai ou faux.

Les réponses pas encore remplies affichent ⏳.

**Ex MB2.1 — Des entiers en base 2 et en base 16 : conversions, une couleur #RRGGBB, les bits pour 5 000 codes, et un mot de 16 bits découpé en deux champs**

In [ ]:
# Ex MB2.1: the values you found on paper
answer_MB2_1a = ...  # a) a text of 8 binary digits, between quotes
answer_MB2_1b = ...  # b) a text of 2 hexadecimal digits, between quotes
answer_MB2_1c = ...  # c) an integer
answer_MB2_1d = ...  # d) a list of 3 integers: red, green, blue
answer_MB2_1e = ...  # e) an integer
answer_MB2_1f = ...  # f) a list of 2 integers: the machine, then the counter
answer_MB2_1g = ...  # g) a text of 4 hexadecimal digits, between quotes

for letter, answer in zip("abcdefg", [answer_MB2_1a, answer_MB2_1b, answer_MB2_1c, answer_MB2_1d, answer_MB2_1e, answer_MB2_1f, answer_MB2_1g]):
    wbm.check(f"MB2.1{letter}", answer)

**Ex MB2.2 — Cent oiseaux, trois milieux, quatre espèces : la PMI des cases, le lift d'une règle, la PPMI, l'espèce vue une seule fois, et l'information mutuelle comme moyenne**

In [ ]:
# Ex MB2.2: the values you found on paper
answer_MB2_2a = ...  # a) a number
answer_MB2_2b = ...  # b) a number (3 decimals)
answer_MB2_2c = ...  # c) a number (3 decimals)
answer_MB2_2d = ...  # d) an integer
answer_MB2_2e = ...  # e) a number (3 decimals)
answer_MB2_2f = ...  # f) an integer
answer_MB2_2g = ...  # g) a number (4 decimals), in nats

for letter, answer in zip("abcdefg", [answer_MB2_2a, answer_MB2_2b, answer_MB2_2c, answer_MB2_2d, answer_MB2_2e, answer_MB2_2f, answer_MB2_2g]):
    wbm.check(f"MB2.2{letter}", answer)

**Ex MB2.3 — Lisser la marge des espèces de la même table (α = 1/3) : la loi lissée, le décalage de la PMI de chaque colonne, et le classement des espèces d'un milieu avant et après**

In [ ]:
# Ex MB2.3: the values you found on paper
answer_MB2_3a = ...  # a) a list of 4 numbers, species 0 to 3
answer_MB2_3b = ...  # b) a list of 4 numbers (4 decimals), species 0 to 3
answer_MB2_3c = ...  # c) a number (3 decimals)
answer_MB2_3d = ...  # d) a list of 3 numbers (3 decimals): woodpecker, thrush, sparrow
answer_MB2_3e = ...  # e) a list of species numbers, by decreasing smoothed PMI
answer_MB2_3f = ...  # f) True or False

for letter, answer in zip("abcdef", [answer_MB2_3a, answer_MB2_3b, answer_MB2_3c, answer_MB2_3d, answer_MB2_3e, answer_MB2_3f]):
    wbm.check(f"MB2.3{letter}", answer)

**Ex MB2.4 — Quatre magasins, trois produits, une table de rang 2 de SVD donnée : les facteurs U S^β et V S^{1−β} pour β = 0, ½ et 1, leur produit, et le cosinus de deux magasins**

In [ ]:
# Ex MB2.4: the values you found on paper
answer_MB2_4a = ...  # a) a list of 2 numbers
answer_MB2_4b = ...  # b) a 4 × 2 matrix, as the list of its rows
answer_MB2_4c = ...  # c) a 3 × 2 matrix, as the list of its rows
answer_MB2_4d = ...  # d) a list of 3 numbers: β = 0, ½, 1
answer_MB2_4e = ...  # e) a list of 3 numbers (4 decimals): β = 0, ½, 1
answer_MB2_4f = ...  # f) True or False

for letter, answer in zip("abcdef", [answer_MB2_4a, answer_MB2_4b, answer_MB2_4c, answer_MB2_4d, answer_MB2_4e, answer_MB2_4f]):
    wbm.check(f"MB2.4{letter}", answer)

**Ex MB2.5 — Une bannière affichée 3 200 fois et cliquée 50 fois : le logit de vraisemblance maximale, puis un non-clic gardé sur 9, le décalage de ln 9 et sa correction**

In [ ]:
# Ex MB2.5: the values you found on paper
answer_MB2_5a = ...  # a) an irreducible fraction, between quotes, like "5/8"
answer_MB2_5b = ...  # b) a number (3 decimals)
answer_MB2_5c = ...  # c) a number (3 decimals)
answer_MB2_5d = ...  # d) a number (3 decimals)
answer_MB2_5e = ...  # e) an irreducible fraction, between quotes
answer_MB2_5f = ...  # f) an irreducible fraction, between quotes

for letter, answer in zip("abcdef", [answer_MB2_5a, answer_MB2_5b, answer_MB2_5c, answer_MB2_5d, answer_MB2_5e, answer_MB2_5f]):
    wbm.check(f"MB2.5{letter}", answer)

**Ex MB2.6 — Yaourt et fromage blanc, nature et sucrés : mesurer le parallélogramme d'une analogie (défaut, rapport ε, borne du cosinus), et dire si d peut l'emporter sur c**

In [ ]:
# Ex MB2.6: the values you found on paper
answer_MB2_6a = ...  # a) a list of 3 numbers
answer_MB2_6b = ...  # b) a number
answer_MB2_6c = ...  # c) a list of 2 numbers (3 decimals): ε, then the bound
answer_MB2_6d = ...  # d) a number (3 decimals)
answer_MB2_6e = ...  # e) a number (4 decimals)
answer_MB2_6f = ...  # f) True or False
answer_MB2_6g = ...  # g) a number (4 decimals)
answer_MB2_6h = ...  # h) a list of 3 numbers (4 decimals): ε, the bound, then cos(q, cream)

for letter, answer in zip("abcdefgh", [answer_MB2_6a, answer_MB2_6b, answer_MB2_6c, answer_MB2_6d, answer_MB2_6e, answer_MB2_6f, answer_MB2_6g, answer_MB2_6h]):
    wbm.check(f"MB2.6{letter}", answer)

**Ex MB2.7 — Des mesures d'animaux en logarithme : la direction de taille estimée sur deux différences, la retirer de trois animaux, et les distances qui restent**

In [ ]:
# Ex MB2.7: the values you found on paper
answer_MB2_7a = ...  # a) a 2 × 2 matrix, as the list of its rows
answer_MB2_7b = ...  # b) a number
answer_MB2_7c = ...  # c) a list of 3 numbers (3 decimals)
answer_MB2_7d = ...  # d) a 3 × 3 matrix, as the list of its rows
answer_MB2_7e = ...  # e) a list of 2 numbers (4 decimals): before, then after
answer_MB2_7f = ...  # f) a number

for letter, answer in zip("abcdef", [answer_MB2_7a, answer_MB2_7b, answer_MB2_7c, answer_MB2_7d, answer_MB2_7e, answer_MB2_7f]):
    wbm.check(f"MB2.7{letter}", answer)

**Ex MB2.11 — Pont : résumer des séries de relevés de longueurs différentes par une moyenne masquée, montrer que le remplissage ne change rien, et compter les comparaisons par paires**

In [ ]:
# Ex MB2.11: the values you found on paper
answer_MB2_11a = ...  # a) a list of 2 numbers: soil temperature, soil humidity
answer_MB2_11b = ...  # b) a list of 2 numbers: soil temperature, soil humidity
answer_MB2_11c = ...  # c) True or False
answer_MB2_11d = ...  # d) an integer
answer_MB2_11e = ...  # e) a number

for letter, answer in zip("abcde", [answer_MB2_11a, answer_MB2_11b, answer_MB2_11c, answer_MB2_11d, answer_MB2_11e]):
    wbm.check(f"MB2.11{letter}", answer)

## Partie A · Ta librairie : la table des PMI ; le logit d'une population et le retrait d'une direction

Fiche §2, §5 et §7. Les trois fonctions de `mymath/cooccurrence.py`, leurs tests, et un usage de chacune sur les données des exercices papier. La cellule de données ci-dessous recopie les données des exercices papier.

In [ ]:
# Tools of the notebook (parts A to D)
import itertools
import math
import re
import time
import warnings

import matplotlib.pyplot as plt
import numpy as np
from scipy.linalg import null_space

from wbm.errors import MymathMissingError
from wbm.nbtools import StopCheck, checked, drawn, filled, fr, own_code, run_pytest_group, verdict

TEST_MB2 = "tests/test_mb2_cooccurrence.py"         # the tests of mymath/cooccurrence.py
REF = wbm.load_mymath("ref", alias="mymath_ref")  # the reference: the wrong versions and the checks rely on it
ref_cooc, ref_svd = REF.cooccurrence, REF.svd_pca
REF_COOC = {f"ref_cooc.{name}": getattr(ref_cooc, name)          # a check stops if your code calls one of them
            for name in ("pmi", "case_control_logit", "remove_direction")}


def shown(value, digits=4):
    """A short text for a result of the learner's code (the check messages): numbers rounded, as lists."""
    if value is None:
        return "None"
    try:
        return str(np.round(np.asarray(value, dtype=float), digits).tolist())[:160]
    except (TypeError, ValueError):
        return repr(value)[:160]


def check_answer(ex_id, value, hint=None):
    """wbm.check(ex_id, value, computed=True). For a list that holds True or False, the message gives only how many
    elements are right, not the place of the first wrong one (it would let you guess the booleans one by one).
    hint(value), when given, may return a targeted « Piste » for a wrong answer (None otherwise)."""
    holds_bool = isinstance(value, (list, tuple)) and any(isinstance(v, (bool, np.bool_)) for v in value)
    if not holds_bool and hint is None:
        return wbm.check(ex_id, value, computed=True)
    result = wbm.check(ex_id, value, computed=True, quiet=True)
    message = result.message
    if holds_bool:
        for cut in (" Le premier faux est le ", " Premier élément faux à l'indice "):
            message = message.split(cut)[0]
    if hint is not None and result.status not in ("correct", "pending", "unknown"):
        try:
            extra = hint(value)
        except Exception:  # noqa: BLE001 - an odd answer: no targeted hint
            extra = None
        if extra:
            message = f"{message} Piste : {extra}"
    icon = {"correct": "✅", "pending": "⏳", "unknown": "❓"}.get(result.status, "❌")
    print(f"{icon} Ex {ex_id} : {message}")
    return result


def passes(ex_id, value):
    """True when value is the hidden answer ex_id (wbm.check, computed=True, nothing printed)."""
    return bool(wbm.check(ex_id, value, computed=True, quiet=True).passed)


def finite_form(values):
    """An array of numbers with -inf written -1000.0, as the hidden answers that hold -inf are recorded (wbm.record
    keeps finite numbers): a list."""
    return np.where(np.isneginf(np.asarray(values, dtype=float)), -1000.0, np.asarray(values, dtype=float)).tolist()


def reporter(results, stop=True):
    """A function report(ok, success, failure) that prints one ✅ or ❌ line, keeps the verdict in results, and (stop)
    ends the check at the first ❌ (StopCheck, caught by checked)."""
    def report(ok, success, failure):
        results.append(bool(ok))
        print(("✅ " + success) if ok else ("❌ " + failure))
        if not ok and stop:
            raise StopCheck
    return report


def as_floats(value):
    """value as a float array, or None when it is not an array of numbers."""
    try:
        return np.asarray(value, dtype=float)
    except (TypeError, ValueError):
        return None


def matches(got, want, rtol=1e-9, atol=1e-12):
    """True when got is an array of numbers of the shape of want and equal to it up to rounding, an infinity equal to
    the same infinity (no error for an odd got)."""
    arr = as_floats(got)
    want = np.asarray(want, dtype=float)
    return arr is not None and arr.shape == want.shape and bool(np.allclose(arr, want, rtol=rtol, atol=atol))


_NUMBER = re.compile(r"[-−]?\d+(?:[.,]\d+)?(?:[eE][-−+]?\d+)?")


def numbers_in(text):
    """The numbers written in a text (a decimal point or comma, a minus sign − or -, an exponent e-07)."""
    found = []
    for token in _NUMBER.findall(text or ""):
        try:
            found.append(float(token.replace("−", "-").replace(",", ".")))
        except ValueError:
            pass
    return found


def gives(text, values, rel=0.01, abs_tol=0.006):
    """True when the text writes one of the values (to 1 % or to 0.006)."""
    return any(math.isclose(x, v, rel_tol=rel, abs_tol=abs_tol) for x in numbers_in(text) for v in values)


def drawn_points(ax):
    """All the points drawn in ax: those of its lines with a marker and of its scatters."""
    points = []
    for line in ax.lines:
        if line.get_marker() in (None, "", " ", "None", "none"):
            continue
        points += list(zip(np.asarray(line.get_xdata(), dtype=float).ravel(),
                           np.asarray(line.get_ydata(), dtype=float).ravel()))
    for collection in ax.collections:
        offsets = np.asarray(collection.get_offsets(), dtype=float)
        if offsets.ndim == 2 and offsets.shape[1] == 2 and type(collection).__name__ == "PathCollection":
            points += [tuple(p) for p in offsets]
    return points


def points_at(ax, xs, ys, rtol=1e-7, atol=1e-9):
    """True when every point (xs[k], ys[k]) is drawn in ax (a line with markers, or a scatter)."""
    points = np.array(drawn_points(ax) or [(np.nan, np.nan)], dtype=float)
    return all(np.any(np.isclose(points[:, 0], x, rtol=rtol, atol=atol) & np.isclose(points[:, 1], y, rtol=rtol, atol=atol))
               for x, y in zip(np.atleast_1d(xs), np.atleast_1d(ys)))


# The data of the paper exercises (02_exercices.md)
BIRDS = np.array([[0.0, 6, 14, 20], [1, 2, 10, 22], [0, 0, 3, 22]])  # MB2.2: 100 birds (counts); rows forêt, prairie, village
P = BIRDS / 100                                                      # their joint law; columns 0 huppe, 1 pic, 2 grive, 3 moineau
SHOPS = np.array([[12.0, 96, 220], [78, 54, 5], [57, 114, 190], [132, 144, 140]])   # MB2.4: shops 1 to 4 (rows), 3 products
BANNER = {"shown": 3200, "clicks": 50, "keep_negative": 1 / 9}       # MB2.5: one non-click in 9 kept
ANIMALS = np.array([[2.0, 3, 4], [3.5, 4.5, 5.5], [5, 3, 4]])        # MB2.7: log2 of (body, tail, ear), animals 1 to 3

### Ex MB2.13 — La table des PMI d'une loi jointe, −∞ sur les cases vides · `pmi` 🔨 ★ ⏱️ 30 min
**Objectif :** programmer la table des PMI d'une loi jointe, avec −∞ sur les cases vides, puis la comparer à ta copie, retrouver l'information mutuelle de M06 comme sa moyenne, et voir ce que deviennent une ligne et une colonne vides.  
**Prérequis :** fiche §2 ; M03, M06 ; Ex MB2.2 · **mymath :** `cooccurrence.py` · **Parcours :** C, E

> **Mode d'emploi `mymath`** (comme en M28). Ouvre ta copie `mon_travail/mymath/cooccurrence.py`, lis la docstring du module (ses conventions, et ce qui est permis) puis celle de la fonction, remplace chaque `raise NotImplementedError(...)` par ton code, **enregistre** le fichier, puis exécute les cellules ci-dessous : la première recharge ta librairie. Dans le notebook, appelle tes fonctions par `mymath.cooccurrence.<nom>` (par exemple `mymath.cooccurrence.pmi(P)`), ou donne-leur un nom court (`pmi = mymath.cooccurrence.pmi`) dans une cellule exécutée après le rechargement. ⚠️ Le squelette n'importe rien : ajoute en tête du fichier `import numpy as np` (et `import math` si tu t'en sers). NumPy et le module `math` sont permis, mais aucune fonction de SciPy (`scipy.special.logit`, `scipy.special.xlogy`) ni de scikit-learn (`mutual_info_score`) : les tests comparent tes résultats aux leurs, et un test lit ton module. Les logarithmes sont népériens ; une loi jointe est donnée par ses **probabilités**, jamais par des comptes ; ne modifie jamais les tableaux reçus. Hors de `mymath`, le notebook peut se servir de tout NumPy et de SciPy, et de tes fonctions de M06 (`mymath.info`), de M12d (`mymath.svd_pca`) et de M28 (`mymath.sampling`).

Écris `pmi(joint)` (fiche §2), en suivant sa docstring : contrôle la table (2-D, non vide, des nombres finis $\ge 0$, pas des booléens, de somme 1 à $10^{-9}$ près) ; calcule les marges $p(x)$ (sommes des lignes) et $p(y)$ (sommes des colonnes), en colonne et en ligne pour qu'elles se diffusent (M20) ; pars d'une table remplie de `-np.inf`, et calcule le logarithme du rapport sur les seules cases de probabilité non nulle, sans avertissement de NumPy ; rends un tableau de flottants de la forme de la table.

**L'usage.** La cellule de données de la partie A donne la table des oiseaux de MB2.2 (`BIRDS`, les comptes) et sa loi jointe `P = BIRDS / 100`. Avec ta fonction :
- a) la table des PMI de `P` (un tableau de forme $(3, 4)$, tel que rendu, $-\infty$ compris), à comparer à ta copie de MB2.2 ;
- b) la moyenne $\sum p(x, y)\operatorname{PMI}(x, y)$ sur les cases non vides, puis vrai ou faux : elle est égale, à $10^{-12}$ près, à `mymath.info.mutual_information(P, base=math.e)` (M06 compte en bits par défaut) (une liste de 2 : le nombre, puis le booléen) ;
- c) vrai ou faux : `pmi(P.T)` est égale à la transposée de `pmi(P)`, à $10^{-12}$ près, avec $-\infty$ aux mêmes places ;
- d) pour la loi $\mathbf{Q} = \begin{pmatrix} 0{,}25 & 0 & 0{,}15 \\ 0 & 0 & 0 \\ 0{,}2 & 0 & 0{,}4 \end{pmatrix}$ (`Q_13`), qui a une ligne et une colonne vides : le nombre de cases de PMI $-\infty$ (un entier).

Ce qu'il faut regarder : que font une ligne vide et une colonne vide ? Que fait à la table des PMI la transposition de la loi ? La moyenne des PMI est-elle l'information mutuelle de M06, et que change l'unité du logarithme ? Ce qu'il faut observer : dans la cellule qui suit la vérification.

In [ ]:
mymath = wbm.load_mymath("learner", missing_ok=True, fallback="ref", chapter="MB2")  # reload your saved file

**Les tests** de `pmi` (lis d'abord la ligne qui suit chaque ❌ : elle dit ce qui était attendu).

In [ ]:
run_pytest_group(TEST_MB2, "test_pmi_ or test_module_uses_")

**L'usage** : la table des oiseaux, sa moyenne, sa transposée, et une loi qui a une ligne et une colonne vides.

In [ ]:
Q_13 = np.array([[0.25, 0.0, 0.15], [0.0, 0.0, 0.0], [0.2, 0.0, 0.4]])
answer_MB2_13a = answer_MB2_13b = answer_MB2_13c = answer_MB2_13d = ...
with wbm.attempt("MB2.13"):          # if you call pmi here: ⏳ while it is not written
    answer_MB2_13a = ...  # a) a (3, 4) array, as returned: pmi(P), -inf included
    answer_MB2_13b = ...  # b) a list of 2: the mean of the PMI under P over the cells of probability > 0 (as computed), then True or False: it equals mymath.info.mutual_information(P, base=math.e) to 1e-12
    answer_MB2_13c = ...  # c) True or False: pmi(P.T) equals the transpose of pmi(P) to 1e-12, with -inf at the same places
    answer_MB2_13d = ...  # d) an integer: the number of cells of pmi(Q_13) equal to -inf

In [ ]:
with checked("MB2.13"):                  # a): compared with the table of the reference, computed now
    want_c13 = ref_cooc.pmi(P)                  # no value is written here
    if not filled(answer_MB2_13a):
        print("⏳ Ex MB2.13a : pas encore fait.")
    else:
        got_c13 = as_floats(answer_MB2_13a)
        if got_c13 is not None and got_c13.shape == (4, 3):
            why_c13 = "la table a une ligne par milieu et une colonne par espèce : la forme (3, 4) de P"
        elif got_c13 is None or got_c13.shape != want_c13.shape:
            why_c13 = f"attendu un tableau de forme {want_c13.shape}, reçu {shown(answer_MB2_13a)[:60]}"
        elif matches(got_c13, want_c13):
            why_c13 = ""
        elif matches(got_c13, want_c13 / math.log(2)):
            why_c13 = "la PMI est en nats : le logarithme népérien, np.log"
        elif matches(got_c13, np.where(np.isinf(want_c13), 0.0, want_c13)):
            why_c13 = "une case de probabilité 0 a une PMI de −∞, pas 0"
        elif matches(got_c13, want_c13 + np.log(P.sum(axis=0))):
            why_c13 = "la PMI divise p(x, y) par p(x) ET par p(y) : ln P(y | x) − ln p(y)"
        else:
            why_c13 = "ce n'est pas la table des PMI de P (relance les tests de pmi)"
        print("✅ Ex MB2.13a : la table des PMI de P, −∞ compris." if not why_c13 else f"❌ Ex MB2.13a : {why_c13}")
for letter, answer in zip("bcd", [answer_MB2_13b, answer_MB2_13c, answer_MB2_13d]):
    check_answer(f"MB2.13{letter}", answer)

**Ce qu'il faut observer** : une ligne vide et une colonne vide ne font ni erreur ni avertissement ; transposer la loi transpose la table ; l'information mutuelle est la moyenne des PMI, et ne dépend que de l'unité du logarithme.

### Ex MB2.14 — Deux briques : le logit d'une population estimé sur un échantillon cas-témoins, et retirer une direction des lignes d'une matrice · `case_control_logit`, `remove_direction` 🔨 ★★ ⏱️ 35 min
**Objectif :** programmer la correction d'un logit estimé sur un échantillon cas-témoins et le retrait d'une direction, puis s'en servir sur la bannière de MB2.5, une étude cas-témoins et les animaux de MB2.7.  
**Prérequis :** fiche §5 et §7 ; M04, M07, M12b, M13a ; Ex MB2.5 et MB2.7 · **mymath :** `cooccurrence.py` · **Parcours :** C, E

> Même mode d'emploi qu'en MB2.13 : écris les fonctions dans ta copie `mon_travail/mymath/cooccurrence.py`, **enregistre**, puis exécute les cellules ci-dessous ; la première recharge ta librairie.

Écris `case_control_logit(n_positive, n_negative, keep_positive=1.0, keep_negative=1.0)` et `remove_direction(X, u)` (fiche §5 et §7), en suivant leurs docstrings. `case_control_logit` : contrôle les deux taux (dans $]0, 1]$, ni `nan` ni booléen), puis les nombres d'exemples (finis, $\ge 0$, pas des booléens, refusés avant la conversion en flottants) ; diffuse-les (`np.broadcast_arrays`) ; traite les trois cas case par case (les deux nombres positifs : le logarithme du rapport, moins celui des taux ; seul $n_1 > 0$ : $+\infty$ ; seul $n_0 > 0$ : $-\infty$), et refuse une case où les deux sont nuls ; rends un `float` quand la forme diffusée est `()`. `remove_direction` : contrôle $\mathbf{X}$ (1-D ou 2-D, non vide, des nombres finis) et $\mathbf{u}$ (1-D, de la longueur des lignes, non nul) ; calcule les coefficients $\mathbf{x}\cdot\mathbf{u}/\mathbf{u}\cdot\mathbf{u}$ de toutes les lignes en un produit matrice-vecteur, puis retire coefficient fois $\mathbf{u}$ à chaque ligne (un produit extérieur, ou la diffusion).

**L'usage.** La cellule de données de la partie A donne la bannière de MB2.5 (`BANNER`) et les animaux de MB2.7 (`ANIMALS`, une ligne par animal). Avec tes fonctions :
- a) la bannière de MB2.5 : `case_control_logit(50, 350, keep_negative=1/9)` (un nombre), à comparer à ta réponse b) de MB2.5 ;
- b) trois bannières d'un même échantillon, de 50, 12 et 0 clics, et de 350, 40 et 25 non-clics gardés, avec le même taux $\frac19$ : les trois logits de la population, en un appel (un tableau de 3 nombres, tel que rendu) ;
- c) une étude cas-témoins garde chaque positif avec la probabilité 0,8 et chaque négatif avec la probabilité 0,025, et compte 30 positifs et 120 négatifs : le logit de la population, puis la probabilité d'un positif dans la population (une liste de 2 nombres) ;
- d) les trois animaux de MB2.7 sans la direction $(1, 1, 1)$, à comparer à ta réponse d) de MB2.7, puis vrai ou faux : la direction $(-2, -2, -2)$ donne le même résultat (à $10^{-12}$ près) (une liste de 10 : les 9 coordonnées, ligne par ligne, puis le booléen) ;
- e) $\lVert\mathbf{x}_1 - \mathbf{x}_3\rVert^2 - \lVert\mathbf{r}_1 - \mathbf{r}_3\rVert^2$, où $\mathbf{r}_i$ est l'animal $i$ après le retrait (un nombre).

Ce qu'il faut regarder : que devient le logit d'une bannière sans clic, avant et après la correction ? Que demande la correction d'un sous-échantillonnage fort des négatifs ? Le retrait d'une direction dépend-il de la longueur de $\mathbf{u}$, et de son sens ? Ce qu'il faut observer : dans la cellule qui suit la vérification.

In [ ]:
mymath = wbm.load_mymath("learner", missing_ok=True, fallback="ref", chapter="MB2")  # reload your saved file

**Les tests** de `case_control_logit`, puis de `remove_direction` (lis d'abord la ligne qui suit chaque ❌ : elle dit ce qui était attendu).

In [ ]:
run_pytest_group(TEST_MB2, "test_case_control_logit_")

In [ ]:
run_pytest_group(TEST_MB2, "test_remove_direction_")

**L'usage** : la bannière, trois bannières en un appel, une étude cas-témoins, puis les animaux.

In [ ]:
answer_MB2_14a = answer_MB2_14b = answer_MB2_14c = answer_MB2_14d = answer_MB2_14e = ...
with wbm.attempt("MB2.14"):          # if you call your functions here: ⏳ while they are not written
    answer_MB2_14a = ...  # a) a number, as computed: case_control_logit(50, 350, keep_negative=1 / 9)
    answer_MB2_14b = ...  # b) an array of 3 numbers, as returned by ONE call: the three banners (-inf included)
    answer_MB2_14c = ...  # c) a list of 2 numbers, as computed: the logit of the population, then the probability of a positive in the population
    answer_MB2_14d = ...  # d) a list of 10: the 9 coordinates of remove_direction(ANIMALS, (1, 1, 1)), row by row (.ravel()), then True or False: the direction (-2, -2, -2) gives the same result to 1e-12
    answer_MB2_14e = ...  # e) a number, as computed: ‖x1 − x3‖² − ‖r1 − r3‖²

In [ ]:
for letter, answer in zip("a", [answer_MB2_14a]):
    check_answer(f"MB2.14{letter}", answer)
with checked("MB2.14"):                  # b): the two finite logits, then the banner without a click
    if not filled(answer_MB2_14b):
        print("⏳ Ex MB2.14b : pas encore fait.")
    elif as_floats(answer_MB2_14b) is None or as_floats(answer_MB2_14b).shape != (3,):
        print(f"❌ Ex MB2.14b : attendu un tableau de 3 logits, rendu par un seul appel, reçu {shown(answer_MB2_14b)}")
    elif not np.isneginf(as_floats(answer_MB2_14b)[2]):
        print("❌ Ex MB2.14b : la troisième bannière n'a aucun clic (n₁ = 0) : son logit vaut −∞, avant comme après la "
              "correction")
    else:
        check_answer("MB2.14b", as_floats(answer_MB2_14b)[:2].tolist())
for letter, answer in zip("cde", [answer_MB2_14c, answer_MB2_14d, answer_MB2_14e]):
    check_answer(f"MB2.14{letter}", answer)

**Ce qu'il faut observer** : une bannière sans clic a un logit de $-\infty$, qui reste $-\infty$ après la correction ; un sous-échantillonnage fort des négatifs se corrige par une simple soustraction ; retirer une direction ne dépend ni de la longueur de $\mathbf{u}$ ni de son sens.

## Partie B · Prédire, dessiner

Fiche §2, §3 et §4. La dispersion d'une PMI estimée, prédite puis mesurée ; des magasins dans le plan des deux facteurs d'une SVD tronquée.

### Ex MB2.15 — Prédire puis mesurer la dispersion de la PMI estimée sur 400 observations, pour une case rare et une case fréquente, avec et sans lissage 🔮 ★★ ⏱️ 30 min
**Objectif :** prévoir ce que le nombre d'observations d'une case fait à sa PMI estimée, et ce que le lissage y change, puis le mesurer.  
**Prérequis :** fiche §2 et §3 ; M02c, M28 ; Ex MB2.2 et MB2.3 · **Parcours :** C

La loi des oiseaux de MB2.2 (`P`) est celle d'une région. Un observateur n'y voit que 400 oiseaux, tirés au hasard selon cette loi, et estime les PMI par `pmi(comptes / 400)`, puis, avec le lissage de MB2.3, par $\ln\frac{\hat p(x, y)}{\hat p(x)\,\hat p_\alpha(y)}$, où $\hat p_\alpha$ est la marge des espèces de l'échantillon tempérée à $T = 3$ (`mymath.sampling.temper`, M28). On suit trois cases (`CELLS_15`) : (prairie, huppe), dont l'espèce est rare et n'est vue qu'en prairie ; (prairie, pic), une case rare d'une espèce présente ailleurs ; (village, moineau), une case fréquente. **Avant d'exécuter quoi que ce soit**, prédis :
- a) la probabilité qu'un échantillon de 400 oiseaux ne contienne aucune huppe, ce qui rend $-\infty$ la PMI estimée de (prairie, huppe) (un nombre, 3 décimales, que tu calcules à la main) ;
- b) celle des trois cases dont la PMI estimée (finie) est la plus dispersée : (A) (prairie, huppe) ; (B) (prairie, pic) ; (C) (village, moineau) (une lettre) ;
- c) avec le lissage, l'écart-type de la PMI estimée de (prairie, huppe) : (A) diminue nettement ; (B) reste à peu près le même ; (C) augmente nettement (une lettre).

Puis, une fois tes prédictions écrites et vérifiées, écris `pmi_samples_15(rng, n_rep, n_obs=400)`, qui tire `n_rep` échantillons, chacun par `rng.multinomial(n_obs, P.ravel())` remis à la forme $(3, 4)$, et rend deux tableaux de forme `(n_rep, 3)` : les PMI estimées des trois cases, dans l'ordre de `CELLS_15`, sans puis avec lissage ($-\infty$ compris) ; `pmi_samples_15` lit `P` au moment de l'appel ; la vérification l'essaie sur des appels cachés, dont un sur une autre loi. Avec `rng = np.random.default_rng(7115)` et 1 000 échantillons, relève :
- d) la proportion des échantillons où la PMI estimée de (prairie, huppe) vaut $-\infty$ (un nombre ; 3 décimales dans ta copie) ;
- e) l'écart-type (`ddof=1`) des valeurs finies de chaque case, sans puis avec lissage, une ligne par case (un tableau de forme $(3, 2)$ ; 3 décimales dans ta copie) ;
- f) les vraies valeurs, $\operatorname{PMI}$ et $\operatorname{PMI}_\alpha$ de la loi `P` de la région, une ligne par case (un tableau de forme $(3, 2)$ ; 3 décimales dans ta copie) ;
- g) vrai ou faux : pour les deux cases rares, la moyenne des estimations lissées finies est plus proche de la vraie $\operatorname{PMI}_\alpha$ que de la vraie PMI.

La vérification de l'expérience ne s'exécute qu'une fois tes trois prédictions écrites. Ce qu'il faut regarder (après l'expérience) : qu'est-ce qui rend une estimation dispersée ? Comment se comportent les deux cases rares, (prairie, huppe) et (prairie, pic), quand la huppe est vue et quand elle ne l'est pas ? Que fait le lissage à la cible, et à la dispersion ? Ce qu'il faut observer : dans la cellule qui suit la vérification.

In [ ]:
CELLS_15 = [(1, 0), (1, 1), (2, 3)]     # (prairie, huppe), (prairie, pic), (village, moineau): (row, column) of P
T_ALPHA_15 = 3                           # the smoothing of MB2.3: alpha = 1/3, a temperature T = 3

📝 **Mon hypothèse** (à écrire **avant** d'exécuter quoi que ce soit) : …

In [ ]:
prediction_MB2_15a = ...  # a) a number (3 decimals), computed by hand BEFORE the experiment: the probability that 400 birds hold no hoopoe
prediction_MB2_15b = ...  # b) "A", "B" or "C": the cell whose estimated PMI (finite) is the most dispersed
prediction_MB2_15c = ...  # c) "A", "B" or "C": with the smoothing, the standard deviation of the estimated PMI of (prairie, huppe)

In [ ]:
for letter, prediction in zip("abc", [prediction_MB2_15a, prediction_MB2_15b, prediction_MB2_15c]):
    wbm.check(f"MB2.15{letter}", prediction)

**L'expérience.** Écris-la une fois tes trois prédictions vérifiées ; sa vérification ne s'exécute qu'à cette condition.

In [ ]:
def pmi_samples_15(rng, n_rep, n_obs=400):
    """n_rep samples of n_obs birds, each drawn by rng.multinomial(n_obs, P.ravel()) (P read at the call) and reshaped to
    (3, 4): two arrays of shape (n_rep, 3), the estimated PMI of the cells CELLS_15 without, then with the smoothing of
    the species margin of the sample (mymath.sampling.temper at T = 3), -inf included."""
    raise NotImplementedError("pmi_samples_15 n'est pas encore écrite (04_indices.md si tu bloques)")


answer_MB2_15d = answer_MB2_15e = answer_MB2_15f = answer_MB2_15g = ...
with wbm.attempt("MB2.15"):          # your function: ⏳ while it is not written
    answer_MB2_15d = ...  # d) a number, as computed: the proportion of samples where the estimated PMI of (prairie, huppe) is -inf (rng = np.random.default_rng(7115), 1 000 samples)
    answer_MB2_15e = ...  # e) a (3, 2) array, as computed: the standard deviations (ddof=1) of the finite values of each cell, without then with the smoothing
    answer_MB2_15f = ...  # f) a (3, 2) array: the true PMI, then the true PMI_alpha, of the law P, one row per cell
    answer_MB2_15g = ...  # g) True or False: for the two rare cells, the mean of the finite smoothed estimates is closer to the true PMI_alpha than to the true PMI

In [ ]:
if any(answer is ... or answer is None for answer in [prediction_MB2_15a, prediction_MB2_15b, prediction_MB2_15c]):
    print("⏳ Écris d'abord tes trois prédictions, puis exécute à nouveau.")
else:
    for letter, answer in zip("defg", [answer_MB2_15d, answer_MB2_15e, answer_MB2_15f, answer_MB2_15g]):
        check_answer(f"MB2.15{letter}", answer)

    HIDDEN_C15 = [([[0.0, 0.06, 0.14, 0.2], [0.01, 0.02, 0.1, 0.22], [0.0, 0.0, 0.03, 0.22]], 7131, 6, 60,
  ('MB2.15h', 'MB2.15i')),
 ([[0.06, 0.11, 0.17, 0.08], [0.03, 0.07, 0.12, 0.09], [0.02, 0.04, 0.06, 0.15]], 7132, 5, 150,
  ('MB2.15j', 'MB2.15k'))]   # hidden calls: (law, seed, n_rep, n_obs) and the sub-IDs of their two arrays in answers.json

    with checked("MB2.15"):
        results_15 = []
        report_15 = reporter(results_15, stop=False)
        law_15 = P                                            # the law of the notebook, put back after the hidden calls
        try:
            for law_c15, seed_c15, n_rep_c15, n_obs_c15, ids_c15 in HIDDEN_C15:
                P = np.array(law_c15)                         # your function reads P at the call
                got_c15 = pmi_samples_15(np.random.default_rng(seed_c15), n_rep_c15, n_obs_c15)
                P = law_15
                label_c15 = (f"un appel caché ({n_rep_c15} échantillons de {n_obs_c15} oiseaux, "
                             f"{'la loi des oiseaux' if np.allclose(law_c15, law_15) else 'une autre loi'})")
                if not isinstance(got_c15, (tuple, list)) or len(got_c15) != 2:
                    report_15(False, "", f"{label_c15} : pmi_samples_15 doit rendre deux tableaux (sans, puis avec lissage), "
                                         f"reçu {shown(got_c15)}")
                    continue
                plain_c15, smooth_c15 = (as_floats(a) for a in got_c15)
                if any(a is None or a.shape != (n_rep_c15, 3) for a in (plain_c15, smooth_c15)):
                    report_15(False, "", f"{label_c15} : chaque tableau a la forme (n_rep, 3) = ({n_rep_c15}, 3), une ligne "
                                         "par échantillon et une colonne par case de CELLS_15")
                    continue
                ok_plain, ok_smooth = passes(ids_c15[0], finite_form(plain_c15)), passes(ids_c15[1], finite_form(smooth_c15))
                if not ok_plain and not np.allclose(law_c15, law_15):
                    again_c15 = pmi_samples_15(np.random.default_rng(seed_c15), n_rep_c15, n_obs_c15)
                    if isinstance(again_c15, (tuple, list)) and matches(again_c15[0], plain_c15):
                        why_c15 = "ta fonction rend la même chose pour deux lois différentes : lis P au moment de l'appel"
                    else:
                        why_c15 = "les PMI estimées sans lissage ne sont pas celles de l'énoncé"
                elif not ok_plain:
                    why_c15 = ("les PMI estimées sans lissage : un échantillon par rng.multinomial(n_obs, P.ravel()), remis à "
                               "la forme (3, 4), puis pmi(comptes / n_obs), lues aux cases de CELLS_15, dans cet ordre")
                elif not ok_smooth:
                    why_c15 = ("les PMI lissées : ln p̂(x, y) − ln p̂(x) − ln p̂_α(y), avec p̂_α la marge des espèces DE "
                               "L'ÉCHANTILLON tempérée à T = 3 (mymath.sampling.temper), −∞ sur une case vide")
                else:
                    why_c15 = ""
                report_15(not why_c15, f"pmi_samples_15 : {label_c15}, comme attendu", f"pmi_samples_15, {label_c15} : {why_c15}")
        finally:
            P = law_15
        verdict("MB2.15", all(results_15) and len(results_15) == len(HIDDEN_C15),
                "ta fonction suit l'énoncé sur des appels cachés aussi.", "relis les lignes ❌ ci-dessus.")

**Ce qu'il faut observer** (après l'expérience) : la dispersion d'une PMI estimée vient surtout du petit nombre d'oiseaux de la case (8 en moyenne pour (prairie, pic), d'où un écart-type d'environ $1/\sqrt8$ ; 88 pour (village, moineau)). (prairie, huppe) est un piège : la huppe n'est vue qu'en prairie, donc le compte de la case est celui de la colonne, et ils se simplifient : l'estimation vaut $\ln\frac{400}{n_{\text{prairie}}}$, très stable, sauf quand aucune huppe n'est vue ($-\infty$). Le lissage décale la cible des colonnes rares vers le bas (c'est son rôle), mais ne réduit pas la dispersion d'une case : il casse même la simplification de la huppe, et son estimation devient aussi dispersée que celle du pic. Pour réduire la dispersion, il faut plus d'observations, ou des pseudo-comptes (M04), que le 🏆 met à l'épreuve.

**Dans ta copie** : pourquoi (prairie, huppe) est-elle si peu dispersée tant que la huppe est vue ? Pourquoi le lissage la rend-il dispersée ? Que faudrait-il pour réduire la dispersion de (prairie, pic) ?

### Ex MB2.16 — Des magasins dans le plan des deux facteurs U₂S₂^β, pour β = 0, ½ et 1 : ce qui s'étire, et les angles qui changent 🎨 ★ ⏱️ 35 min
**Objectif :** dessiner les lignes d'une table dans le plan de U₂S₂^β pour trois β, et voir ce que la répartition des valeurs singulières change à la forme du nuage et aux angles.  
**Prérequis :** fiche §4 ; M07, M12d ; Ex MB2.4 · **Parcours :** C

Le notebook fournit `M_16`, les ventes hebdomadaires (en dizaines d'articles) de cinq produits (pain, fruits, œufs, vis, peinture) dans dix magasins : trois épiceries, trois quincailleries et quatre bazars, et `SHOPS_16`, leurs noms.

Écris `factor_rows_16(M, beta, k=2)`, qui rend $\mathbf{U}_k\mathbf{S}_k^\beta$ par ta fonction `mymath.svd_pca.svd` (M12d, avec sa convention de signe), puis `draw_factors_16(axes, M, betas, names)`, qui reçoit une ligne de trois axes et, pour chaque $\beta$ de `betas` (un axe chacun), place les lignes de $\mathbf{U}_2\mathbf{S}_2^\beta$ dans le plan de ses deux colonnes : un point et le nom de chaque ligne, pris dans `names`, une même échelle sur les deux axes (`ax.set_aspect("equal")`), des noms d'axes, et en titre $\beta$ et le rapport des longueurs (normes) de la première et de la seconde colonne ; elle rend la liste des trois tableaux tracés. Les deux fonctions dessinent dans les axes reçus, sans ouvrir de figure.

La vérification appelle `draw_factors_16` avec $\beta = 0$, $\frac12$ et 1, affiche la figure, compare tes tableaux aux réponses enregistrées, et essaie tes fonctions sur une autre table, d'autres $\beta$ et $k = 3$ (des appels cachés). Relève :
- a) les trois plus grandes valeurs singulières de `M_16` (une liste de 3 nombres ; 2 décimales dans ta copie) ;
- b) le cosinus de l'épicerie A et de la quincaillerie A entre les lignes de $\mathbf{U}_2\mathbf{S}_2^\beta$, pour $\beta = 0$, $\frac12$ et 1 (une liste de 3 nombres ; 3 décimales dans ta copie) ;
- c) le même pour les bazars A et B ;
- d) le rapport des longueurs des deux colonnes de $\mathbf{U}_2\mathbf{S}_2^\beta$, pour les trois $\beta$ (une liste de 3 nombres ; 3 décimales dans ta copie) ;
- e) vrai ou faux : les cosinus entre les lignes de $\mathbf{U}_2\mathbf{S}_2$ sont ceux des lignes de $\mathbf{M}_2$, l'approximation de rang 2 (`mymath.svd_pca.low_rank_approximation`), à $10^{-9}$ près ;
- f) vrai ou faux : ce sont ceux des lignes de `M_16` elle-même, à $10^{-9}$ près.

Ce qu'il faut regarder : que devient le nuage quand $\beta$ monte, et le long de quel axe ? Les magasins semblent-ils se rapprocher ou s'éloigner les uns des autres ? Quels angles $\beta = 1$ garde-t-il ? Ce qu'il faut observer : dans la cellule qui suit la vérification.

In [ ]:
SHOPS_16 = ["épicerie A", "épicerie B", "épicerie C", "quincaillerie A", "quincaillerie B", "quincaillerie C",
            "bazar A", "bazar B", "bazar C", "bazar D"]
PRODUCTS_16 = ["pain", "fruits", "œufs", "vis", "peinture"]
M_16 = np.array([[66.0, 46, 42, 3, 5], [53, 40, 30, 10, 6], [83, 66, 50, 6, 3],          # weekly sales (tens of items):
                 [2, 1, 0, 22, 16], [4, 2, 1, 29, 25], [0, 1, 1, 14, 13],              # one row per shop of SHOPS_16,
                 [34, 28, 20, 18, 14], [48, 38, 29, 11, 9], [14, 10, 9, 26, 19],       # one column per product
                 [29, 20, 18, 15, 10]])

In [ ]:
def factor_rows_16(M, beta, k=2):
    """U_k S_k^beta for the compact SVD of M by mymath.svd_pca.svd (M12d, its sign convention): an array of shape
    (number of rows of M, k)."""
    raise NotImplementedError("factor_rows_16 n'est pas encore écrite (04_indices.md si tu bloques)")


def draw_factors_16(axes, M, betas, names):
    """axes: a row of 3 axes. For each beta of betas (one axes each): the rows of factor_rows_16(M, beta) as points in the
    plane of its two columns, the name of each row (names) next to its point, the same scale on both axes
    (ax.set_aspect("equal")), axis labels, and a title giving beta and the ratio of the lengths (norms) of the first and
    the second column. Returns the list of the 3 arrays drawn. Opens no figure."""
    raise NotImplementedError("draw_factors_16 n'est pas encore écrite (04_indices.md si tu bloques)")


answer_MB2_16a = answer_MB2_16b = answer_MB2_16c = answer_MB2_16d = answer_MB2_16e = answer_MB2_16f = ...
with wbm.attempt("MB2.16"):          # your functions: ⏳ while they are not written
    answer_MB2_16a = ...  # a) a list of 3 numbers, as computed: the three largest singular values of M_16
    answer_MB2_16b = ...  # b) a list of 3 numbers, as computed: cos(épicerie A, quincaillerie A) between the rows of U2 S2^beta, for beta = 0, 0.5, 1
    answer_MB2_16c = ...  # c) a list of 3 numbers, as computed: the same for bazar A and bazar B
    answer_MB2_16d = ...  # d) a list of 3 numbers, as computed: the ratio of the lengths of the two columns of U2 S2^beta, for beta = 0, 0.5, 1
    answer_MB2_16e = ...  # e) True or False: the cosines between the rows of U2 S2 are those of the rows of M2, to 1e-9
    answer_MB2_16f = ...  # f) True or False: they are those of the rows of M_16 itself, to 1e-9

In [ ]:
for letter, answer in zip("abcdef", [answer_MB2_16a, answer_MB2_16b, answer_MB2_16c, answer_MB2_16d, answer_MB2_16e, answer_MB2_16f]):
    check_answer(f"MB2.16{letter}", answer)

HIDDEN_C16 = {'table': [[18.0, 12, 3, 1], [22, 15, 5, 2], [4, 3, 16, 11], [2, 5, 21, 14], [12, 9, 10, 7], [9, 4, 13, 12]],
 'names': ['kiosque 1', 'kiosque 2', 'kiosque 3', 'kiosque 4', 'kiosque 5', 'kiosque 6'],
 'betas': [0.25, 0.6, 0.9],
 'beta': 0.4,
 'ids': ['MB2.16h', 'MB2.16i']}   # a hidden table, its names, three other betas, and the sub-IDs of the arrays expected


def factors_why_c16(axes, M, betas, names, arrays, sub_id):
    """What is wrong in a call of draw_factors_16 (French phrases): the arrays returned, then the three panels."""
    if not isinstance(arrays, (list, tuple)) or len(arrays) != len(betas):
        return [f"draw_factors_16 doit rendre la liste des {len(betas)} tableaux tracés, reçu {shown(arrays)}"]
    arrays = [as_floats(a) for a in arrays]
    shape = (len(M), 2)
    if any(a is None or a.shape != shape for a in arrays):
        return [f"chaque tableau tracé a une ligne par ligne de M et deux colonnes : la forme {shape}"]
    problems = []
    if not passes(sub_id, [a.tolist() for a in arrays]):
        flipped = any(passes(sub_id, [(a * signs).tolist() for a in arrays]) for signs in ([-1, 1], [1, -1], [-1, -1]))
        problems.append("au signe près d'une colonne : prends la SVD de mymath.svd_pca.svd, qui suit la convention de "
                        "signe de M12d" if flipped else
                        "les tableaux tracés ne sont pas U₂S₂^β pour les β reçus (dans l'ordre de betas), à partir de la "
                        "SVD de la table reçue")
    for k, (ax, beta, a) in enumerate(zip(axes, betas, arrays)):
        missing = []
        if not points_at(ax, a[:, 0], a[:, 1]):
            missing.append("un point par ligne, à ses deux coordonnées (ax.scatter, ou ax.plot avec un marqueur)")
        if not set(names) <= {text.get_text().strip() for text in ax.texts}:
            missing.append("le nom de chaque ligne près de son point (ax.annotate ou ax.text)")
        if ax.get_aspect() not in (1.0, "equal"):
            missing.append('la même échelle sur les deux axes (ax.set_aspect("equal"))')
        if not ax.get_xlabel() or not ax.get_ylabel():
            missing.append("des noms d'axes")
        if not gives(ax.get_title(), [float(np.linalg.norm(a[:, 0]) / np.linalg.norm(a[:, 1]))]):
            missing.append("en titre, le rapport des longueurs des deux colonnes (un nombre)")
        if missing:
            problems.append(f"panneau {k + 1} (β = {fr(beta, 2)}) : il manque " + " ; ".join(missing))
    return problems


with checked("MB2.16"):
    results_16 = []
    report_16 = reporter(results_16, stop=False)
    fig_16, axes_16 = plt.subplots(1, 3, figsize=(16, 5.2))
    try:
        result_16 = []
        opened_16 = drawn(lambda: result_16.append(draw_factors_16(axes_16, M_16, (0, 0.5, 1), SHOPS_16)))
        problems_16 = factors_why_c16(axes_16, M_16, (0, 0.5, 1), SHOPS_16, result_16[0], "MB2.16g")
        report_16(not problems_16, "draw_factors_16 (β = 0, ½ et 1) : les trois tableaux, les points, les noms, l'échelle, "
                  "les noms d'axes et les titres", "draw_factors_16 : " + " ; ".join(problems_16))
        fig_c16, axes_c16 = plt.subplots(1, 4)             # the 4th axes is the current one: plt.scatter would draw there
        try:
            hidden_c16 = []
            opened_16 += drawn(lambda: hidden_c16.append(draw_factors_16(axes_c16[:3], np.array(HIDDEN_C16["table"]),
                                                                          HIDDEN_C16["betas"], HIDDEN_C16["names"])))
            problems_c16 = factors_why_c16(axes_c16[:3], HIDDEN_C16["table"], HIDDEN_C16["betas"], HIDDEN_C16["names"],
                                           hidden_c16[0], HIDDEN_C16["ids"][0])
            elsewhere_16 = bool(axes_c16[3].lines or axes_c16[3].collections or axes_c16[3].texts or axes_c16[3].get_title())
        finally:
            plt.close(fig_c16)
        report_16(not problems_c16, "un appel caché de draw_factors_16 (une autre table, d'autres β) est juste aussi",
                  "un appel caché de draw_factors_16 (une autre table, d'autres β) : " + " ; ".join(problems_c16))
        rows_c16 = as_floats(factor_rows_16(np.array(HIDDEN_C16["table"]), HIDDEN_C16["beta"], k=3))
        shape_c16 = (len(HIDDEN_C16["table"]), 3)
        if rows_c16 is None or rows_c16.shape != shape_c16:
            why_c16 = (f"attendu U₃S₃^β, de forme {shape_c16} (k est le nombre de colonnes gardées), reçu "
                       f"{None if rows_c16 is None else rows_c16.shape}")
        elif passes(HIDDEN_C16["ids"][1], rows_c16.tolist()):
            why_c16 = ""
        elif any(passes(HIDDEN_C16["ids"][1], (rows_c16 * signs).tolist())
                 for signs in itertools.product([1, -1], repeat=3) if min(signs) < 0):
            why_c16 = "au signe près d'une colonne : prends la SVD de mymath.svd_pca.svd, qui suit la convention de signe de M12d"
        else:
            why_c16 = "ce n'est pas U₃S₃^β, les k = 3 premières colonnes de U multipliées par les puissances β de s"
        report_16(not why_c16, "factor_rows_16 sur la table cachée, avec k = 3 et un autre β, comme attendu",
                  f"factor_rows_16 sur la table cachée, avec k = 3 : {why_c16}")
        report_16(not opened_16 and not elsewhere_16, "tes fonctions dessinent dans les axes reçus",
                  "tes fonctions doivent dessiner dans les axes reçus (ax.scatter, ax.set_title…), sans ouvrir de figure ni "
                  "dessiner dans l'axe courant (plt.scatter, plt.title…)")
    except BaseException:
        plt.close(fig_16)
        raise
    fig_16.tight_layout()
    plt.show()
    verdict("MB2.16", all(results_16), "tes facteurs et tes trois panneaux sont justes.", "relis les lignes ❌ ci-dessus.")

**Ce qu'il faut observer** : quand $\beta$ monte, le nuage s'étire le long du premier axe, celui du profil moyen ; les magasins semblent alors plus proches les uns des autres ; $\beta = 1$ garde les angles de la table approchée, pas ceux de la table.

**Après la figure**, dans ta copie : quels magasins se rapprochent quand $\beta$ monte, et pourquoi ? Quel $\beta$ choisirais-tu pour comparer les magasins par le cosinus, et pour comparer les produits ?

## Partie C · Déboguer, tester

Fiche §2, §4 et §7. Trois erreurs classiques, qui passent une vérification mal choisie ; un test de propriété de `pmi`. Les versions fausses et le code du collègue s'appuient sur la référence (`ref_cooc`, `ref_svd`), pour ne pas dépendre de ton code.

### Ex MB2.17 — Chasse à l'erreur : une PMI calculée sans le total, des facteurs U S^β et V S^β, une direction retirée sans diviser par u·u 🐛 ★ ⏱️ 30 min
**Objectif :** reconnaître trois erreurs classiques sur la PMI d'une table de comptes, la répartition des valeurs singulières et le retrait d'une direction, qui passent une vérification mal choisie, puis les corriger.  
**Prérequis :** fiche §2, §4 et §7 ; Ex MB2.2, MB2.4 et MB2.7 · **Parcours :** C, E

Un collègue a écrit trois fonctions, chacune accompagnée d'une vérification qui passe (cellule fournie) :
1. `pmi_from_counts(C)` doit rendre la table des PMI d'une table de **comptes** ; elle calcule `np.log(C / np.outer(C.sum(1), C.sum(0)))` sur les cases non nulles, et $-\infty$ ailleurs. Il l'a vérifiée sur une loi jointe : elle redonne `pmi`.
2. `split_factors(U, s, Vt, beta)` doit rendre les deux facteurs $\mathbf{U}\mathbf{S}^\beta$ et $\mathbf{V}\mathbf{S}^{1-\beta}$ d'une SVD compacte ; elle rend `U * s ** beta, Vt.T * s ** beta`. Il l'a vérifiée avec $\beta = \frac12$ sur les magasins de MB2.4 : le produit des deux facteurs redonne $\mathbf{M}$.
3. `remove_direction_bad(X, u)` doit retirer la direction $\mathbf{u}$ de chaque ligne de $\mathbf{X}$ ; elle rend `X - np.outer(X @ u, u)`. Il l'a vérifiée avec $\mathbf{u} = (0 ;\ 0{,}6 ;\ 0{,}8)$ sur les animaux de MB2.7 : elle redonne `remove_direction`.

À faire : trouve et corrige chaque erreur ; les fonctions corrigées s'appellent `pmi_from_counts_fixed`, `split_factors_fixed` et `remove_direction_fixed`, avec les mêmes arguments. Puis réponds :
- a) pour la table des comptes des oiseaux de MB2.2 (`BIRDS`), $\operatorname{PMI}(\text{forêt}, \text{pic})$ par `pmi_from_counts_fixed`, puis par `pmi_from_counts` (une liste de 2 nombres) ;
- b) pour les magasins de MB2.4 (`SHOPS`), avec $\beta = 1$ : la case (1, 1) (en haut à gauche) du produit des deux facteurs de `split_factors_fixed`, puis de `split_factors`, la SVD calculée par `mymath.svd_pca.svd` (une liste de 2 nombres) ;
- c) l'animal 1 de MB2.7 sans la direction $(1, 1, 1)$, par `remove_direction_fixed`, puis par `remove_direction_bad` (une liste de 6 nombres).

Dans ta copie : pour chacune des trois erreurs, pourquoi la vérification du collègue ne la voyait pas. La vérification essaie tes corrections sur des cas cachés : des tables de comptes de totaux variés, d'autres $\beta$ et d'autres matrices, des directions de normes diverses.

In [ ]:
# The colleague's code
def pmi_from_counts(C):
    """The PMI table of a table of COUNTS C (-inf on the empty cells)."""
    C = np.asarray(C, dtype=float)
    out = np.full(C.shape, -np.inf)
    nonzero = C > 0
    out[nonzero] = np.log(C[nonzero] / np.outer(C.sum(axis=1), C.sum(axis=0))[nonzero])
    return out


def split_factors(U, s, Vt, beta):
    """The two factors U S^beta and V S^(1 - beta) of a compact SVD (U, s, Vt)."""
    return U * s ** beta, Vt.T * s ** beta


def remove_direction_bad(X, u):
    """X without the direction u: each row x becomes x minus its component along u."""
    X, u = np.asarray(X, dtype=float), np.asarray(u, dtype=float)
    return X - np.outer(X @ u, u)


# The colleague's checks: all pass
print("(1)", bool(np.allclose(pmi_from_counts(P), ref_cooc.pmi(P))), "(attendu True : sur une loi jointe)")
U_17, s_17, Vt_17 = ref_svd.svd(SHOPS)
A_17, B_17 = split_factors(U_17, s_17, Vt_17, 0.5)
print("(2)", bool(np.allclose(A_17 @ B_17.T, SHOPS)), "(attendu True : avec beta = 0.5)")
u_17 = np.array([0.0, 0.6, 0.8])
print("(3)", bool(np.allclose(remove_direction_bad(ANIMALS, u_17), ref_cooc.remove_direction(ANIMALS, u_17))),
      "(attendu True : avec u = (0 ; 0,6 ; 0,8))")

In [ ]:
def pmi_from_counts_fixed(C):
    """The PMI table of a table of COUNTS C (-inf on the empty cells)."""
    raise NotImplementedError("pmi_from_counts_fixed n'est pas encore écrite (04_indices.md si tu bloques)")


def split_factors_fixed(U, s, Vt, beta):
    """The two factors U S^beta and V S^(1 - beta) of a compact SVD (U, s, Vt)."""
    raise NotImplementedError("split_factors_fixed n'est pas encore écrite (04_indices.md si tu bloques)")


def remove_direction_fixed(X, u):
    """X without the direction u: each row x becomes x minus its component along u."""
    raise NotImplementedError("remove_direction_fixed n'est pas encore écrite (04_indices.md si tu bloques)")


answer_MB2_17a = answer_MB2_17b = answer_MB2_17c = ...
with wbm.attempt("MB2.17"):          # your three functions: ⏳ while they are not written
    answer_MB2_17a = ...  # a) a list of 2 numbers, as computed: PMI(forêt, pic) of the counts BIRDS by pmi_from_counts_fixed, then by the WRONG pmi_from_counts
    answer_MB2_17b = ...  # b) a list of 2 numbers, as computed: the top left cell of the product of the two factors of SHOPS with beta = 1, by split_factors_fixed, then by the WRONG split_factors (the SVD by mymath.svd_pca.svd)
    answer_MB2_17c = ...  # c) a list of 6 numbers, as computed: animal 1 of ANIMALS without the direction (1, 1, 1), by remove_direction_fixed, then by the WRONG remove_direction_bad

In [ ]:
for letter, answer in zip("abc", [answer_MB2_17a, answer_MB2_17b, answer_MB2_17c]):
    check_answer(f"MB2.17{letter}", answer)

CASES_C17 = {'laws': [[[0.2, 0.05], [0.15, 0.6]], [[0.1, 0.0, 0.35], [0.2, 0.05, 0.3]]],
 'counts': [[[3, 0, 7], [12, 5, 0], [0, 9, 4]], [[30, 12], [0, 0], [7, 51], [22, 3]],
            [[260, 41, 7, 0, 19], [88, 300, 12, 5, 0]]],
 'factors': [{'M': [[4.0, 1, 7], [2, 9, 3], [8, 5, 1], [6, 2, 2], [1, 3, 9]], 'beta': 0.0},
             {'M': [[4.0, 1, 7], [2, 9, 3], [8, 5, 1], [6, 2, 2], [1, 3, 9]], 'beta': 0.3},
             {'M': [[2.0, 0, 5, 1, 3, 7], [6, 4, 1, 0, 2, 2], [1, 8, 3, 5, 0, 4], [3, 2, 2, 6, 9, 1]],
              'beta': 1.0}],
 'directions': [{'X': [[1.5, -2.5, 0.5], [3, 1, -1], [0, 2.5, 4], [-2, -1, 3]], 'u': [0.1, -0.2, 0.05]},
                {'X': [[3.0, 1], [-1, 4], [2, 2], [0, -3], [5, -2]], 'u': [3.5, -8.0]},
                {'X': [[1.0, 0, 2, -1], [3, 3, 0, 1], [-2, 1, 1, 4]], 'u': [1.5, -1, 0, 2.5]}]}   # the hidden cases of the three corrected functions
rng_c17 = np.random.default_rng()       # at each run: new factors for the counts

results_17, parts_17 = [], []


def report_17(ok, success, failure):
    results_17.append(bool(ok))
    print(("✅ " + success) if ok else ("❌ " + failure))


def call_17(part, name, function, *args, **kwargs):
    """function(*args, **kwargs); an error of the learner's code becomes one ❌ line that names the function (the check of
    this part stops there, the final verdict stays)."""
    try:
        return function(*args, **kwargs)
    except (NotImplementedError, MymathMissingError):
        raise
    except Exception as exc:  # noqa: BLE001 - the learner's error, reported with the name of the function
        report_17(False, "", f"({part}) {name} : ton code lève {type(exc).__name__} : {str(exc)[:160]}")
        parts_17.append(part)
        raise StopCheck from None


with checked("MB2.17"):                                       # (1) pmi_from_counts_fixed
    wrong_17 = []
    for law_c17 in CASES_C17["laws"]:                         # joint laws (sum 1): the PMI of pmi
        law_c17 = np.array(law_c17)
        got_c17 = as_floats(call_17(1, "pmi_from_counts_fixed", pmi_from_counts_fixed, law_c17.copy()))
        if got_c17 is None or got_c17.shape != law_c17.shape or not matches(got_c17, ref_cooc.pmi(law_c17)):
            wrong_17.append("sur une loi jointe (de somme 1), ta fonction doit redonner pmi")
    for counts_c17 in CASES_C17["counts"]:                    # tables of counts: k C describes the same law as C
        counts_c17 = np.array(counts_c17)
        before_c17 = counts_c17.copy()
        got_c17 = as_floats(call_17(1, "pmi_from_counts_fixed", pmi_from_counts_fixed, counts_c17))
        if got_c17 is None or got_c17.shape != counts_c17.shape:
            wrong_17.append(f"attendu une table de la forme des comptes, {counts_c17.shape}, reçu {shown(got_c17)[:60]}")
            continue
        if not np.array_equal(counts_c17, before_c17):
            wrong_17.append("ta fonction modifie la table reçue : travaille sur une copie")
            continue
        k_c17 = float(rng_c17.uniform(0.2, 50.0))
        scaled_c17 = as_floats(call_17(1, "pmi_from_counts_fixed", pmi_from_counts_fixed, k_c17 * counts_c17))
        if not matches(scaled_c17, got_c17, rtol=1e-9, atol=1e-9):
            wrong_17.append(f"une table de comptes de total {int(counts_c17.sum())} : multiplier tous les comptes par un "
                            "même nombre ne change pas la loi, donc pas la PMI ; ta fonction doit rendre la même table "
                            "pour C et pour k·C")
    report_17(not wrong_17, "(1) pmi_from_counts_fixed : des lois jointes, et trois tables de comptes de totaux variés, "
              "multipliées par un facteur tiré au hasard", "(1) pmi_from_counts_fixed : " + " ; ".join(list(dict.fromkeys(wrong_17))[:2]))
    parts_17.append(1)

with checked("MB2.17"):                                       # (2) split_factors_fixed
    wrong_17 = []
    for case_c17 in CASES_C17["factors"]:
        U_c17, s_c17, Vt_c17 = ref_svd.svd(np.array(case_c17["M"]))
        beta_c17 = case_c17["beta"]
        got_c17 = call_17(2, "split_factors_fixed", split_factors_fixed, U_c17, s_c17, Vt_c17, beta_c17)
        if not isinstance(got_c17, (tuple, list)) or len(got_c17) != 2:
            wrong_17.append(f"split_factors_fixed doit rendre les deux facteurs, reçu {shown(got_c17)[:60]}")
            continue
        A_c17, B_c17 = (as_floats(f) for f in got_c17)
        shape_A, shape_B = U_c17.shape, Vt_c17.T.shape
        if A_c17 is None or B_c17 is None or A_c17.shape != shape_A or B_c17.shape != shape_B:
            wrong_17.append(f"les deux facteurs ont les formes {shape_A} (une ligne par ligne de M) et {shape_B} (une "
                            "ligne par colonne de M)")
            continue
        if not matches(A_c17, split_factors(U_c17, s_c17, Vt_c17, beta_c17)[0]):
            wrong_17.append("le premier facteur U S^β était juste : ne le change pas")
            continue
        if not np.allclose(A_c17 @ B_c17.T, (U_c17 * s_c17) @ Vt_c17, rtol=1e-9, atol=1e-9 * float(s_c17.max())):
            wrong_17.append(f"avec β = {fr(beta_c17, 2)} et une autre matrice, le produit des deux facteurs doit redonner "
                            "U S Vᵀ : les puissances de S des deux facteurs s'ajoutent")
    report_17(not wrong_17, "(2) split_factors_fixed : d'autres matrices, avec β = 0, 0,3 et 1",
              "(2) split_factors_fixed : " + " ; ".join(list(dict.fromkeys(wrong_17))[:2]))
    parts_17.append(2)

with checked("MB2.17"):                                       # (3) remove_direction_fixed
    wrong_17 = []
    for case_c17 in CASES_C17["directions"]:
        X_c17, u_c17 = np.array(case_c17["X"]), np.array(case_c17["u"])
        before_c17 = X_c17.copy()
        R_c17 = as_floats(call_17(3, "remove_direction_fixed", remove_direction_fixed, X_c17, u_c17))
        size_c17 = float(np.abs(X_c17).max()) * float(np.linalg.norm(u_c17))
        if R_c17 is None or R_c17.shape != X_c17.shape:
            wrong_17.append(f"attendu un tableau de la forme de X, {X_c17.shape}, reçu {shown(R_c17)[:60]}")
        elif not np.array_equal(X_c17, before_c17):
            wrong_17.append("ta fonction modifie le tableau reçu : travaille sur une copie")
        elif not np.allclose(R_c17 @ u_c17, 0.0, atol=1e-9 * size_c17):
            wrong_17.append(f"avec une direction u de norme {fr(float(np.linalg.norm(u_c17)), 2)}, chaque ligne du résultat "
                            "doit être orthogonale à u : on retire (x·u / u·u) u")
        elif not np.allclose((X_c17 - R_c17) @ null_space(u_c17[None, :]), 0.0, atol=1e-9 * size_c17):
            wrong_17.append("on ne retire à chaque ligne qu'un multiple de u : ce qui est orthogonal à u ne change pas")
    report_17(not wrong_17, "(3) remove_direction_fixed : trois directions cachées, de normes diverses",
              "(3) remove_direction_fixed : " + " ; ".join(list(dict.fromkeys(wrong_17))[:2]))
    parts_17.append(3)

if len(set(parts_17)) == 3:
    verdict("MB2.17", all(results_17), "les trois fonctions corrigées passent tous les cas.", "relis les lignes ❌ ci-dessus.")

**Après la vérification**, dans ta copie : pour chaque erreur, pourquoi la vérification du collègue ne la voyait-elle pas, et quelle vérification l'aurait vue ?

### Ex MB2.18 — Un test de propriété de `pmi` sur des tables tirées au hasard : la borne, la transposition, et la moyenne qui redonne l'information mutuelle de M06 🛠️ ★ ⏱️ 35 min
**Objectif :** écrire les contrôles d'un test de propriété de la table des PMI, sur des lois tirées au hasard, et voir ce qu'il attrape et ce qu'il ne peut pas attraper.  
**Prérequis :** fiche §2 ; M06 ; Ex MB2.13 · **Parcours :** C

La cellule fournie donne la boucle du test, `pmi_property_test_18(n_trials, rng, func=None, rtol=1e-9, checks=None)` : quand `func` vaut `None`, elle prend `mymath.cooccurrence.pmi` **au moment de l'appel**, et, quand `checks` vaut `None`, ta fonction `failed_checks_18` ; pour chacun des `n_trials` essais, dans cet ordre, elle tire `m = int(rng.integers(1, 6))`, `n = int(rng.integers(1, 6))`, `W = rng.random((m, n)) ** 2`, puis, si `rng.random() < 0.4`, met à 0 les cases où `rng.random((m, n)) < 0.3` (des cases vides) ; si `W` est nulle, elle met `W[0, 0] = 1` ; elle divise `W` par sa somme, appelle `checks(P, f, rtol)`, où `P` est la loi tirée et `f` la fonction testée, et ajoute 1 au compteur de chaque nom rendu ; elle rend le dictionnaire des trois compteurs d'essais `"bound"`, `"transpose"` et `"mean"`.

Écris les contrôles, `failed_checks_18(P, func, rtol=1e-9)`, qui rend l'**ensemble** des noms des contrôles qui échouent pour la loi `P` (`set()` si tout passe) : avec `T = func(P)`,
- `"bound"` échoue si `T` n'a pas la forme de `P`, si une case de probabilité non nulle a une PMI non finie, ou si l'une d'elles dépasse $\min(-\ln p(x), -\ln p(y))$ de plus de $\texttt{rtol}\cdot\max(1, \lvert\text{borne}\rvert)$ ;
- `"transpose"`, si `func(P.T)` n'a pas la forme de `T.T`, n'a pas ses $-\infty$ aux mêmes places, ou s'en écarte ailleurs de plus de `rtol` (`np.allclose` avec `rtol` et `atol` égaux à `rtol`) ;
- `"mean"`, si $\sum p(x, y)\,T(x, y)$, sur les cases non vides, s'écarte de `mymath.info.mutual_information(P, base=math.e)` de plus de $\texttt{rtol}\cdot\max(1, \lvert\text{moyenne}\rvert)$ (ou n'est pas finie).

Une exception levée par `func` fait échouer les trois contrôles. La cellule fournie donne aussi quatre versions fausses de `pmi` : `decimal` (le logarithme en base 10), `conditional` ($\ln P(Y = y \mid X = x)$, sans diviser par $p(y)$), `zeros_to_zero` (0 au lieu de $-\infty$ sur les cases vides) et `lift` (le lift, sans le logarithme, 0 sur les cases vides). Puis :
- a) avec `rng = np.random.default_rng(7118)` et 300 essais, avec ta fonction `pmi` et tes contrôles : vrai ou faux, les trois compteurs sont nuls ;
- b) avec le même départ (un nouveau `default_rng(7118)` chaque fois, 300 essais) et chacune des quatre versions fausses, dans l'ordre `decimal`, `conditional`, `zeros_to_zero`, `lift` : les trois compteurs, dans l'ordre `"bound"`, `"transpose"`, `"mean"` (une liste de 4 listes de 3 entiers) ;
- c) dans ta copie : lesquelles passent le test, ou passent l'un des contrôles, et pourquoi c'est normal ; qu'est-ce qui, dans le dépôt, vérifie ce qu'elles ont de faux ?

La vérification lance aussi ton test sur deux départs cachés de 40 essais, avec la référence (compteurs nuls) et avec quatre versions fausses cachées, qui faussent chacune une ou deux des propriétés ; elle remplace aussi un instant `mymath.cooccurrence.pmi` par l'une d'elles, pour l'appel sans `func`. Ce qu'il faut retenir : ce test ajoute à ceux de `tests/test_mb2_cooccurrence.py` des tables de formes et de cases vides tirées au hasard ; chacun de ses contrôles attrape certaines erreurs et en laisse passer d'autres ; un test se juge aux erreurs qu'il attrape, et à celles qu'il ne peut pas attraper.

In [ ]:
def decimal(P):
    """WRONG: the PMI in base 10."""
    return ref_cooc.pmi(P) / math.log(10)


def conditional(P):
    """WRONG: ln P(Y = y | X = x), without dividing by p(y); -inf on the empty cells."""
    P = np.asarray(P, dtype=float)
    out = np.full(P.shape, -np.inf)
    rows, cols = np.nonzero(P > 0)
    out[rows, cols] = np.log(P[rows, cols]) - np.log(P.sum(axis=1)[rows])
    return out


def zeros_to_zero(P):
    """WRONG: 0 instead of -inf on the empty cells."""
    T = ref_cooc.pmi(P)
    T[np.isneginf(T)] = 0.0
    return T


def lift(P):
    """WRONG: the lift p(x, y) / (p(x) p(y)), without the logarithm; 0 on the empty cells."""
    P = np.asarray(P, dtype=float)
    out = np.zeros(P.shape)
    rows, cols = np.nonzero(P > 0)
    out[rows, cols] = P[rows, cols] / (P.sum(axis=1)[rows] * P.sum(axis=0)[cols])
    return out


# The loop of the property test (provided): the draws, in a fixed order, and the counters
def pmi_property_test_18(n_trials, rng, func=None, rtol=1e-9, checks=None):
    """n_trials random joint laws, drawn in this order: m = int(rng.integers(1, 6)), n = int(rng.integers(1, 6)),
    W = rng.random((m, n)) ** 2, then, if rng.random() < 0.4, W set to 0 where rng.random((m, n)) < 0.3; W[0, 0] = 1 if W
    is zero; the law is W / W.sum(). func (the function tested) defaults to mymath.cooccurrence.pmi, checks to your
    failed_checks_18, both looked up at the call; each name returned by checks(law, func, rtol) adds 1 to its counter.
    Returns the counters "bound", "transpose", "mean"."""
    f = mymath.cooccurrence.pmi if func is None else func
    chk = failed_checks_18 if checks is None else checks
    counts = {"bound": 0, "transpose": 0, "mean": 0}
    for _ in range(n_trials):
        m = int(rng.integers(1, 6))
        n = int(rng.integers(1, 6))
        W = rng.random((m, n)) ** 2
        if rng.random() < 0.4:
            W[rng.random((m, n)) < 0.3] = 0.0
        if W.sum() == 0:
            W[0, 0] = 1.0
        for name in chk(W / W.sum(), f, rtol):
            if name not in counts:
                raise ValueError(f"les contrôles rendent {name!r} : les noms sont 'bound', 'transpose' et 'mean'")
            counts[name] += 1
    return counts

In [ ]:
def failed_checks_18(P, func, rtol=1e-9):
    """The set of the names of the checks that fail for the joint law P, among "bound", "transpose" and "mean" (set() when
    all pass); func is the function tested."""
    raise NotImplementedError("failed_checks_18 n'est pas encore écrite (04_indices.md si tu bloques)")


answer_MB2_18a = answer_MB2_18b = ...
with wbm.attempt("MB2.18"):          # your checks and your library: ⏳ while they are not written
    answer_MB2_18a = ...  # a) True or False: the three counters are 0 (default_rng(7118), 300 trials, your pmi, your checks)
    answer_MB2_18b = ...  # b) a list of 4 lists of 3 integers: the counters ("bound", "transpose", "mean") for decimal, conditional, zeros_to_zero, lift (a new default_rng(7118) each time, 300 trials)

In [ ]:
for letter, answer in zip("ab", [answer_MB2_18a, answer_MB2_18b]):
    check_answer(f"MB2.18{letter}", answer)

PMI_C18 = ref_cooc.pmi                    # captured: the hidden versions never call a function replaced later


def half_c18(P):
    """WRONG: half the PMI (the mean is wrong; the bound and the transposition hold)."""
    return 0.5 * PMI_C18(P)


def double_c18(P):
    """WRONG: twice the PMI (the bound and the mean are wrong; the transposition holds)."""
    return 2.0 * PMI_C18(P)


def finite_tall_c18(P):
    """WRONG for a table with more rows than columns: -1e300 instead of -inf on the empty cells."""
    T = PMI_C18(P)
    if T.shape[0] > T.shape[1]:
        T[np.isneginf(T)] = -1e300
    return T


def tall_shift_c18(P):
    """WRONG for a table with more rows than columns: each finite PMI moved by 0.2 (i - mean of i under p(x)), which
    keeps the mean."""
    P = np.asarray(P, dtype=float)
    T = PMI_C18(P)
    if T.shape[0] > T.shape[1]:
        i = np.arange(T.shape[0], dtype=float)
        shift = 0.2 * (i - np.sum(P.sum(axis=1) * i))
        T = np.where(np.isfinite(T), T + shift[:, None], T)
    return T


BROKEN_C18 = {"half_c18": ("mean",), "double_c18": ("bound", "mean"), "finite_tall_c18": ("transpose",),
              "tall_shift_c18": ("bound", "transpose")}           # the properties each hidden version can break
HIDDEN_C18 = [(7141,
  {'half_c18': {'mean': 'MB2.18d'},
   'double_c18': {'bound': 'MB2.18e', 'mean': 'MB2.18f'},
   'finite_tall_c18': {'transpose': 'MB2.18g'},
   'tall_shift_c18': {'bound': 'MB2.18h', 'transpose': 'MB2.18i'}}),
 (7142,
  {'half_c18': {'mean': 'MB2.18j'},
   'double_c18': {'bound': 'MB2.18k', 'mean': 'MB2.18l'},
   'finite_tall_c18': {'transpose': 'MB2.18m'},
   'tall_shift_c18': {'bound': 'MB2.18n', 'transpose': 'MB2.18o'}})]   # two hidden starts, and the sub-IDs of the counters expected for each version

with checked("MB2.18"):
    results_18 = []
    report_18 = reporter(results_18, stop=False)
    N_C18 = 40
    PROPERTIES_C18 = {"bound": "la borne min(−ln p(x), −ln p(y)) sur les cases non vides",
                      "transpose": "que func(P.T) redonne la transposée (−∞ aux mêmes places, le reste à rtol près)",
                      "mean": "que la moyenne des PMI sous P redonne l'information mutuelle de M06, en nats"}

    def run_c18(label, seed, **kwargs):
        """The loop of the notebook with your failed_checks_18, on a hidden start: its counters."""
        try:
            return pmi_property_test_18(N_C18, np.random.default_rng(seed), checks=failed_checks_18, **kwargs)
        except (NotImplementedError, MymathMissingError):
            raise
        except Exception as exc:  # noqa: BLE001 - the learner's error
            hint = (" ; failed_checks_18 doit rendre un ensemble de noms (set() si tout passe) : as-tu oublié le return ?"
                    if isinstance(exc, TypeError) and "NoneType" in str(exc) else "")
            print(f"❌ Ex MB2.18 : failed_checks_18 ({label}) lève {type(exc).__name__} : {str(exc)[:200]}{hint}")
            raise StopCheck from None

    def compare_c18(label, got, version, ids):
        """One ✅ when the three counters are those expected; else one ❌ per counter, naming the property."""
        wrong = []
        for key in ("bound", "transpose", "mean"):
            if version is None or key not in BROKEN_C18[version]:            # 0 by construction
                if got[key] != 0:
                    wrong.append((key, f"vaut {got[key]}, attendu 0 : "
                                       + ("avec une fonction juste, aucun contrôle ne compte : ton contrôle compte trop "
                                          "(sa tolérance rtol·max(1, |…|), ce qu'il compare)" if version is None else
                                          "cette version ne fausse pas cette propriété : ton contrôle compte trop (sa "
                                          "tolérance rtol·max(1, |…|), ce qu'il compare)")))
            elif not passes(ids[key], got[key]):
                wrong.append((key, ("vaut 0 : tes contrôles ne vérifient pas " + PROPERTIES_C18[key]
                                    + ", sur la fonction reçue (func, pas celle de mymath)") if got[key] == 0 else
                                   (f"vaut {got[key]}, ce n'est pas le compte attendu : relis ce que ce contrôle compare, "
                                    "et sur quelles cases")))
        if not wrong:
            report_18(True, f"{label} : les trois compteurs comme attendu", "")
        for key, why in wrong:
            report_18(False, "", f"{label} : le compteur {key!r} {why}")

    for start_c18, (seed_c18, ids_c18) in enumerate(HIDDEN_C18, start=1):
        compare_c18(f"la référence, départ caché {start_c18}", run_c18("la référence", seed_c18, func=ref_cooc.pmi),
                    None, {})
        for number_c18, name_c18 in enumerate(BROKEN_C18, start=1):
            label_c18 = f"une version fausse cachée n° {number_c18}, départ caché {start_c18}"
            compare_c18(label_c18, run_c18(label_c18, seed_c18, func=globals()[name_c18]), name_c18, ids_c18[name_c18])
    learner_pmi_c18 = mymath.cooccurrence.pmi                # replaced an instant, put back even if the check fails
    try:
        mymath.cooccurrence.pmi = half_c18
        got_c18 = run_c18("func=None", HIDDEN_C18[0][0])
    finally:
        mymath.cooccurrence.pmi = learner_pmi_c18
    compare_c18("la version fausse n° 1 mise un instant à la place de mymath.cooccurrence.pmi, sans func", got_c18,
                "half_c18", HIDDEN_C18[0][1]["half_c18"])
    verdict("MB2.18", all(results_18), "tes contrôles comptent juste sur la référence et sur les versions fausses "
            "cachées.", "relis les lignes ❌ ci-dessus.")

**Dans ta copie** : c) lesquelles des quatre versions fausses passent le test, ou l'un des contrôles, pourquoi c'est normal, et ce qui, dans le dépôt, vérifie ce qu'elles ont de faux.

**Après la vérification**, dans ta copie : quel contrôle faudrait-il ajouter pour attraper `zeros_to_zero`, et pourquoi ne s'agit-il plus d'une propriété des seules cases non vides ?

## Partie D · Défi

Fiche §2 et §3. Les cases les plus associées d'une table cachée, retrouvées à partir de comptes, sur la table du notebook et trois tables cachées.

### Ex MB2.19 — Défi : retrouver, sur 3 000 observations, les cinq paires les plus associées d'une table cachée de 10 × 10, sans se laisser piéger par les cases rares 🏆 ★★★ ⏱️ 45 min
**Objectif :** retrouver les cases de plus grande PMI d'une table cachée à partir de comptes, en se protégeant des cases rares, et le vérifier sur des tables cachées.  
**Prérequis :** fiche §2 et §3 ; M04 ; Ex MB2.13 et MB2.15 · **Parcours :** C

Une table jointe cachée de $10 \times 10$ décrit, pour les articles vendus par un hypermarché, le rayon de l'article (10 rayons, en lignes) et la tranche horaire de la vente (10 tranches, en colonnes) ; quelques rayons et quelques tranches sont rares. Les « cinq paires les plus associées » sont les cinq cases de plus grande PMI de la table cachée, toutes de probabilité nettement non nulle. La cellule fournie donne `COUNTS_19`, les comptes, de forme $(10, 10)$, de 3 000 ventes tirées selon une telle table, et `pairs_found_19(pairs)`, qui compte combien des 5 paires `(ligne, colonne)` données sont parmi les cinq paires cachées, sans rien montrer d'elles.

Écris `top_pairs_19(counts)`, qui reçoit une table de comptes de 3 000 observations et rend une liste de 5 paires `(ligne, colonne)`, numérotées à partir de 0. Tout NumPy et SciPy sont permis, et ta `pmi`.

**Critère de réussite :** au moins 4 des 5 paires trouvées pour la table du notebook, et pour chacune de trois tables cachées de la vérification, dont les lignes et les colonnes sont renumérotées à chaque exécution, et dont les 3 000 ventes sont tirées à nouveau, en moins de 10 secondes en tout. La vérification recalcule aussi ta réponse a) avec ta fonction, et refuse une fonction qui rend les mêmes paires pour deux tables. Relève :
- a) le nombre de paires que ta méthode trouve pour la table du notebook, `pairs_found_19(top_pairs_19(COUNTS_19))` (un entier) ;
- b) le nombre de paires que trouve l'estimation directe, les cinq plus grandes valeurs de `pmi(COUNTS_19 / 3000)` (un entier) ;
- c) les comptes des cinq cases que choisit l'estimation directe, par PMI estimée décroissante (une liste de 5 entiers).

Point de départ : regarde les cases que choisit l'estimation directe, et leurs comptes ; la fiche (§2 et §3) et M04 (les pseudo-comptes) donnent des idées, et la méthode est libre.

In [ ]:
COUNTS_19 = np.array([[47, 3, 47, 81, 166, 40, 25, 0, 2, 56], [24, 1, 35, 72, 37, 33, 29, 0, 3, 203],
 [41, 5, 37, 70, 90, 62, 168, 1, 1, 103], [3, 0, 1, 3, 1, 1, 1, 0, 0, 2], [2, 0, 1, 2, 0, 1, 1, 0, 0, 3],
 [26, 1, 43, 64, 46, 119, 29, 2, 2, 83], [42, 2, 35, 72, 59, 39, 28, 0, 2, 80],
 [26, 1, 146, 46, 34, 33, 20, 3, 1, 45], [2, 0, 0, 5, 0, 2, 0, 0, 0, 5], [34, 2, 47, 92, 37, 38, 38, 0, 1, 64]])   # 3 000 sales: rows the 10 departments, columns the 10 time slots


def pairs_found_19(pairs):
    """The number of the 5 pairs (row, column) given that are among the five most associated pairs of the hidden table
    of COUNTS_19: each pair is compared with the hidden answer by wbm.check, which shows nothing of it."""
    pairs = [tuple(int(v) for v in pair) for pair in pairs]
    if len(pairs) != 5 or len(set(pairs)) != 5 or not all(0 <= i < 10 and 0 <= j < 10 for i, j in pairs):
        raise ValueError(f"pairs_found_19 attend 5 paires (ligne, colonne) distinctes, de 0 à 9, reçu {pairs}")
    return sum(any(passes(f"MB2.19{letter}", list(pair)) for letter in "klmno") for pair in pairs)

In [ ]:
def top_pairs_19(counts):
    """The 5 most associated pairs (row, column), numbered from 0, of the hidden joint law whose counts (3 000
    observations, a (10, 10) array) are given: a list of 5 pairs."""
    raise NotImplementedError("top_pairs_19 n'est pas encore écrite (04_indices.md si tu bloques)")


answer_MB2_19a = answer_MB2_19b = answer_MB2_19c = ...
with wbm.attempt("MB2.19"):          # your function: ⏳ while it is not written
    answer_MB2_19a = ...  # a) an integer: pairs_found_19(top_pairs_19(COUNTS_19))
    answer_MB2_19b = ...  # b) an integer: the pairs found by the direct estimate, the 5 largest values of pmi(COUNTS_19 / 3000)
    answer_MB2_19c = ...  # c) a list of 5 integers: the counts of the 5 cells of the direct estimate, by decreasing estimated PMI

In [ ]:
for letter, answer in zip("bc", [answer_MB2_19b, answer_MB2_19c]):
    check_answer(f"MB2.19{letter}", answer)

HIDDEN_C19 = [[[0.02343933, 0.01776726, 0.0228772, 0.05477653, 0.01360586, 0.01498585, 0.00040789, 0.01483155, 0.00049256,
   0.00062896],
  [0.01440803, 0.01901452, 0.03220812, 0.01497376, 0.01392315, 0.01932932, 0.00041711, 0.01529879, 0.00050369,
   0.00064318],
  [0.00059462, 0.00075755, 0.00078267, 0.0005578, 0.00046913, 0.0006475, 1.368e-05, 0.00046225, 1.652e-05,
   2.11e-05],
  [0.01356559, 0.09767129, 0.01753175, 0.01758726, 0.0111437, 0.02255479, 0.0004137, 0.01424636, 0.00049957,
   0.00063792],
  [0.00064218, 0.00081814, 0.00084527, 0.00060242, 0.00050665, 0.00069928, 1.478e-05, 0.00049922, 1.784e-05,
   2.278e-05],
  [0.0150216, 0.01490329, 0.01608453, 0.01263663, 0.01188289, 0.01460993, 0.00033943, 0.04148274, 0.00040988,
   0.00052339],
  [0.01584827, 0.01742887, 0.02530427, 0.01442376, 0.01334482, 0.01719263, 0.0003277, 0.00910711, 0.00039572,
   0.00050531],
  [0.00908397, 0.01351107, 0.01693685, 0.01047774, 0.00788608, 0.04514483, 0.00021955, 0.00630005, 0.00026512,
   0.00033854],
  [0.00047156, 0.00060077, 0.00062069, 0.00044236, 0.00037204, 0.00051349, 1.085e-05, 0.00036658, 1.31e-05,
   1.673e-05],
  [0.01304767, 0.01597846, 0.06391543, 0.01584881, 0.00946869, 0.01586601, 0.00030996, 0.00997196, 0.0003743,
   0.00047796]],
 [[0.00064319, 0.01338623, 0.01695378, 0.00071785, 0.00079079, 0.01733166, 0.01369635, 0.01668881, 0.00991749,
   0.06421865],
  [1.646e-05, 0.00028877, 0.00038684, 1.837e-05, 2.023e-05, 0.0003387, 0.00044237, 0.00036091, 0.00027088,
   0.00041618],
  [0.00086238, 0.06233041, 0.0211701, 0.00096248, 0.00106027, 0.01536952, 0.02513404, 0.0148745, 0.01236335,
   0.01615674],
  [0.00072287, 0.01213269, 0.01668553, 0.00080678, 0.00088875, 0.01132603, 0.01863753, 0.01662402, 0.00968591,
   0.01834994],
  [2.68e-05, 0.00047028, 0.00063002, 2.991e-05, 3.295e-05, 0.0005516, 0.00072044, 0.00058777, 0.00044115,
   0.00067779],
  [0.00077421, 0.01392699, 0.01783676, 0.00086407, 0.00095187, 0.04795799, 0.02261106, 0.02023693, 0.01235107,
   0.01930699],
  [0.00043201, 0.00597454, 0.03802201, 0.00048215, 0.00053114, 0.0078489, 0.0115857, 0.00982057, 0.00731358,
   0.00947406],
  [0.00065461, 0.01215226, 0.01321376, 0.0007306, 0.00080483, 0.01226573, 0.0185578, 0.01615649, 0.01125924,
   0.0174221],
  [1.917e-05, 0.00033643, 0.00045069, 2.14e-05, 2.357e-05, 0.0003946, 0.00051538, 0.00042047, 0.00031559,
   0.00048486],
  [0.00093845, 0.01820035, 0.02370216, 0.00104738, 0.0011538, 0.02228162, 0.07616099, 0.02503501, 0.01652275,
   0.02323831]],
 [[0.00058732, 0.01865951, 0.02232591, 0.00707999, 0.01525546, 0.01014261, 0.02022667, 0.00071094, 0.00054382,
   0.00988399],
  [0.00079501, 0.02189883, 0.02918546, 0.01190725, 0.02464096, 0.01228646, 0.02162368, 0.00096234, 0.00073612,
   0.01515247],
  [2.531e-05, 0.00084708, 0.00084894, 0.00040188, 0.00086076, 0.00041184, 0.00075132, 3.064e-05, 2.344e-05,
   0.00043331],
  [0.0005697, 0.02402862, 0.02024349, 0.00815993, 0.01537879, 0.00809292, 0.01797852, 0.00068961, 0.0005275,
   0.03876904],
  [0.0005909, 0.06804847, 0.01932988, 0.00948275, 0.02210195, 0.01042497, 0.01958738, 0.00071527, 0.00054713,
   0.01059325],
  [0.00038353, 0.01580499, 0.0122359, 0.00682861, 0.01330092, 0.0063181, 0.03999663, 0.00046425, 0.00035512,
   0.00656056],
  [0.00056786, 0.0168785, 0.01410871, 0.03798479, 0.02402863, 0.00743725, 0.01419957, 0.00068739, 0.0005258,
   0.01138761],
  [3.585e-05, 0.00119954, 0.00120217, 0.00056909, 0.0012189, 0.0005832, 0.00106394, 4.339e-05, 3.319e-05,
   0.0006136],
  [2.145e-05, 0.00071776, 0.00071933, 0.00034052, 0.00072934, 0.00034896, 0.00063662, 2.596e-05, 1.986e-05,
   0.00036716],
  [0.00080665, 0.02496144, 0.09125263, 0.01163313, 0.03056664, 0.01194224, 0.02833058, 0.00097644, 0.00074691,
   0.0131394]]]   # three hidden joint laws of 10 × 10 (rows and columns renumbered at each run)
rng_c19 = np.random.default_rng()       # at each run: a new numbering of the rows and columns, and new sales


def pairs_why_c19(pairs):
    """'' when pairs is a list of 5 distinct pairs (row, column) of integers from 0 to 9, else the reason."""
    try:
        pairs = [tuple(int(v) for v in pair) for pair in pairs]
    except (TypeError, ValueError):
        return f"top_pairs_19 doit rendre une liste de 5 paires (ligne, colonne), reçu {shown(pairs)}"
    if len(pairs) != 5 or any(len(p) != 2 for p in pairs) or len(set(pairs)) != 5:
        return f"top_pairs_19 doit rendre 5 paires (ligne, colonne) distinctes, reçu {shown(pairs)}"
    if not all(0 <= v < 10 for p in pairs for v in p):
        return "les lignes et les colonnes se numérotent de 0 à 9"
    return ""


def top5_c19(law):
    """The five cells of largest PMI of a hidden joint law (the truth of a hidden table, not a method)."""
    T = ref_cooc.pmi(law)
    return {(int(i), int(j)) for i, j in zip(*np.unravel_index(np.argsort(-T, axis=None, kind="stable")[:5], T.shape))}


def rare_hint_c19(counts, pairs):
    """A hint when the pairs chosen hold cells of few sales (their PMI is very dispersed, MB2.15)."""
    few = [int(counts[i, j]) for i, j in pairs if counts[i, j] <= 5]
    return (f" ; {len(few)} des cases choisies n'ont que {', '.join(map(str, sorted(few)))} vente(s) : leur PMI estimée "
            "est très dispersée (MB2.15)") if few else ""


with checked("MB2.19"):
    results_19 = []
    report_19 = reporter(results_19, stop=False)
    elapsed_19 = 0.0
    start_19 = time.perf_counter()
    mine_19 = top_pairs_19(COUNTS_19.copy())
    elapsed_19 += time.perf_counter() - start_19
    why_19 = pairs_why_c19(mine_19)
    found_19 = pairs_found_19(mine_19) if not why_19 else 0
    report_19(not why_19 and found_19 >= 4, f"la table du notebook : {found_19} paires sur 5",
              f"la table du notebook : {why_19 or f'{found_19} paires sur 5, il en faut au moins 4'}"
              + ("" if why_19 else rare_hint_c19(COUNTS_19, mine_19)))
    if not filled(answer_MB2_19a):
        print("⏳ Ex MB2.19a : pas encore fait.")
    elif not why_19:
        report_19(answer_MB2_19a == found_19, "Ex MB2.19a : ta réponse est le score de ta fonction",
                  "Ex MB2.19a : ta réponse n'est pas le score que rend ta fonction maintenant, "
                  "pairs_found_19(top_pairs_19(COUNTS_19)) : exécute à nouveau ta cellule")
    answers_c19 = [sorted(map(tuple, np.asarray(mine_19, dtype=int).tolist())) if not why_19 else None]
    for k_c19, law_c19 in enumerate(HIDDEN_C19, start=1):
        rows_c19, cols_c19 = rng_c19.permutation(10), rng_c19.permutation(10)
        law_c19 = np.array(law_c19)[np.ix_(rows_c19, cols_c19)]
        law_c19 = law_c19 / law_c19.sum()                   # written to 8 decimals: summed to 1 again
        counts_c19 = rng_c19.multinomial(3000, law_c19.ravel()).reshape(10, 10)
        start_19 = time.perf_counter()
        got_c19 = top_pairs_19(counts_c19.copy())
        elapsed_19 += time.perf_counter() - start_19
        why_c19 = pairs_why_c19(got_c19)
        if not why_c19 and sorted(map(tuple, np.asarray(got_c19, dtype=int).tolist())) in answers_c19:
            why_c19 = ("ta fonction rend les mêmes paires pour deux tables différentes : elle doit les calculer à partir "
                       "des comptes reçus")
        if why_c19:
            report_19(False, "", f"table cachée {k_c19} : {why_c19}")
            continue
        got_c19 = [tuple(int(v) for v in p) for p in got_c19]
        answers_c19.append(sorted(got_c19))
        score_c19 = len(set(got_c19) & top5_c19(law_c19))
        report_19(score_c19 >= 4, f"table cachée {k_c19} : {score_c19} paires sur 5",
                  f"table cachée {k_c19} : {score_c19} paires sur 5, il en faut au moins 4"
                  + rare_hint_c19(counts_c19, got_c19))
    report_19(elapsed_19 < 10, f"en {fr(elapsed_19, 2)} s pour les quatre tables",
              f"trop lent : {fr(elapsed_19, 1)} s pour les quatre tables, il faut moins de 10 s")
    verdict("MB2.19", all(results_19) and len(results_19) >= 5, "critère atteint : au moins 4 paires sur 5 pour la table "
            "du notebook et pour les trois tables cachées, en moins de 10 secondes.",
            "critère pas encore atteint : relis les lignes ❌ ci-dessus.")

**Pour réfléchir** : pourquoi l'estimation directe choisit-elle des cases de deux ou trois ventes ? Que devient ta méthode avec 30 000 ventes, et avec 300 ? Que se passerait-il si une vraie paire était dans un rayon rare ?

## ✅ Bilan

**Auto-évaluation** (note-toi de 0 à 3 dans `mon_travail/suivi/auto_evaluation.md`) :
1. Sais-tu écrire un entier en base 2 et en base 16, et découper un mot de bits en champs ?
2. Sais-tu calculer la table des PMI d'une loi jointe, la lire comme un lift, et retrouver l'information mutuelle comme sa moyenne ?
3. Sais-tu dire pourquoi les cases rares ont les PMI les plus grandes et les plus dispersées, et ce que fait le lissage ?
4. Sais-tu répartir les valeurs singulières entre deux facteurs, et dire ce qui en dépend ?
5. Sais-tu corriger le logit d'un échantillon cas-témoins ?
6. Sais-tu retirer une direction d'un nuage, et dire quelles distances changent ?

**La suite** : le chapitre bonus B2 de workbookIA, qui applique ces outils aux tokenizers, à la PPMI suivie d'une SVD, à word2vec et aux embeddings de phrases.